# nf-binder-design setup notebook

This notebook helps you prepare input files and configuration for `nf-binder-design` workflows:
- RFdiffusion (`--method rfd`)
- BindCraft (`--method bindcraft`)
- BoltzGen (`--method boltzgen`)
- or all methods together

Features:
- Load a target structure from a local PDB file or from RCSB PDB ID
- Visualize in 3D
- Define contigs and hotspots
- Choose sensible method presets
- Export `params.json` and BoltzGen YAML
- Generate ready-to-run Nextflow commands

In [6]:
# If needed, uncomment and run once
# %pip install biopython py3Dmol ipywidgets requests pyyaml plotly anywidget nglview

In [7]:
import json
import re
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple

import requests
import yaml
import py3Dmol
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
from Bio.PDB import PDBParser

try:
    import nglview as nv
except Exception:
    nv = None

WORKDIR = Path.cwd()
OUTDIR = WORKDIR / 'nfb_setup_output'
OUTDIR.mkdir(parents=True, exist_ok=True)
(OUTDIR / 'inputs').mkdir(parents=True, exist_ok=True)

print(f'Working directory: {WORKDIR}')
print(f'Output directory:  {OUTDIR}')
print(f'Inputs directory:  {OUTDIR / "inputs"}')
if nv is None:
    print('Note: nglview is unavailable in this kernel; 3D click-to-hotspot is disabled.')

Working directory: /home/jmobbs/code/nf_binder_project
Output directory:  /home/jmobbs/code/nf_binder_project/nfb_setup_output
Inputs directory:  /home/jmobbs/code/nf_binder_project/nfb_setup_output/inputs


## 1) Load target PDB and inspect

In [8]:
_state: Dict[str, object] = {
    'pdb_text': '',
    'pdb_path': '',
    'chain_ranges': {},
    'residue_map': {},
    'ligand_map': {},
    'selected_chain': '',
    'selected_contig_residues': set(),
    'selected_contig_ligands': set(),
    'selected_hotspots': set(),
    'ngl_contig_anchor': None,
}

WATER_NAMES = {'HOH', 'WAT', 'H2O'}

def fetch_pdb_from_rcsb(pdb_id: str) -> str:
    pdb_id = pdb_id.strip().upper()
    if not re.match(r'^[A-Z0-9]{4}$', pdb_id):
        raise ValueError('PDB ID should be 4 alphanumeric characters.')
    url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.text

def load_pdb_text(source_mode: str, file_path: str, pdb_id: str) -> str:
    if source_mode == 'local_file':
        if not file_path.strip():
            raise ValueError('Please enter a PDB file path, or switch Source to RCSB PDB ID.')
        p = Path(file_path).expanduser().resolve()
        if not p.exists():
            raise FileNotFoundError(f'File not found: {p}')
        if p.is_dir():
            raise IsADirectoryError(f'Path is a directory, not a PDB file: {p}')
        if not p.is_file():
            raise ValueError(f'Path is not a regular file: {p}')
        return p.read_text()
    return fetch_pdb_from_rcsb(pdb_id)

def parse_chain_data(
    pdb_path: Path,
) -> Tuple[Dict[str, Tuple[int, int]], Dict[str, List[int]], Dict[str, List[Tuple[str, int]]]]:
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('target', str(pdb_path))
    chain_ranges: Dict[str, Tuple[int, int]] = {}
    residue_map: Dict[str, List[int]] = {}
    ligand_map: Dict[str, List[Tuple[str, int]]] = {}
    for model in structure:
        for chain in model:
            residues = sorted({r.get_id()[1] for r in chain.get_residues() if r.get_id()[0] == ' '})
            if residues:
                residue_map[chain.id] = residues
                chain_ranges[chain.id] = (min(residues), max(residues))
            ligands: List[Tuple[str, int]] = []
            for r in chain.get_residues():
                hetflag, resseq, _icode = r.get_id()
                if hetflag == ' ':
                    continue
                resn = str(r.get_resname()).strip().upper()
                if resn in WATER_NAMES:
                    continue
                ligands.append((resn, int(resseq)))
            if ligands:
                ligand_map[chain.id] = sorted(set(ligands), key=lambda x: (x[1], x[0]))
        break
    return chain_ranges, residue_map, ligand_map

def selected_contigs_as_ranges(residues: Set[int]) -> List[Tuple[int, int]]:
    if not residues:
        return []
    vals = sorted(residues)
    ranges: List[Tuple[int, int]] = []
    start = vals[0]
    prev = vals[0]
    for r in vals[1:]:
        if r == prev + 1:
            prev = r
            continue
        ranges.append((start, prev))
        start = r
        prev = r
    ranges.append((start, prev))
    return ranges

def parse_contigs_for_chain(contig_text: str, chain_id: str) -> Set[int]:
    residues: Set[int] = set()
    text = contig_text.strip().strip('[]')
    if not text or not chain_id:
        return residues

    pattern = rf'{re.escape(chain_id)}(\d+)-(\d+)'
    for start_s, end_s in re.findall(pattern, text):
        a = int(start_s)
        b = int(end_s)
        if a > b:
            a, b = b, a
        residues.update(range(a, b + 1))
    return residues

def parse_hotspot_tokens(hotspot_res: str) -> List[Tuple[str, int]]:
    out: List[Tuple[str, int]] = []
    for token in [x.strip() for x in hotspot_res.split(',') if x.strip()]:
        m = re.match(r'^([A-Za-z])(\d+)$', token)
        if m:
            out.append((m.group(1), int(m.group(2))))
    return out

def make_ligand_token(chain: str, resn: str, resi: int) -> str:
    return f'{chain}:{resn}:{resi}'

def parse_ligand_token(token: str) -> Optional[Tuple[str, str, int]]:
    parts = token.split(':')
    if len(parts) != 3:
        return None
    chain = parts[0].strip()
    resn = parts[1].strip().upper()
    try:
        resi = int(parts[2])
    except Exception:
        return None
    if not chain or not resn:
        return None
    return chain, resn, resi

def format_hotspots(tokens: Set[str]) -> str:
    ordered = sorted(tokens, key=lambda t: (t[0], int(t[1:])))
    return ','.join(ordered)

def format_ligands(tokens: Set[str]) -> str:
    parsed = [p for p in (parse_ligand_token(t) for t in tokens) if p is not None]
    ordered = sorted(parsed, key=lambda x: (x[0], x[2], x[1]))
    return ','.join([f'{c}{r}:{n}' for c, n, r in ordered])

def format_contigs(chain_id: str, ranges: List[Tuple[int, int]], binder_len: str) -> str:
    if not chain_id or not ranges:
        return ''
    receptor = '/'.join([f'{chain_id}{a}-{b}' for a, b in ranges]) + '/0'
    if binder_len.strip():
        return '[' + receptor + ' ' + binder_len.strip() + ']'
    return '[' + receptor + ']'

def show_viewer_py3dmol(
    pdb_text: str,
    selected_chain: str = '',
    hotspot_res: str = '',
    contig_ranges: Optional[List[Tuple[int, int]]] = None,
    show_ligands: bool = False,
    selected_ligands: Optional[Set[str]] = None,
):
    v = py3Dmol.view(width=900, height=550)
    v.addModel(pdb_text, 'pdb')

    if selected_chain:
        v.setStyle({}, {'cartoon': {'color': 'lightgray'}})
        v.addStyle({'chain': selected_chain}, {'cartoon': {'color': 'spectrum'}})
    else:
        v.setStyle({}, {'cartoon': {'color': 'spectrum'}})

    if show_ligands:
        v.addStyle({'hetflag': True}, {'stick': {'radius': 0.18, 'color': 'orange'}})
        v.setStyle({'hetflag': True, 'resn': ['HOH', 'WAT', 'H2O']}, {})

    if selected_chain and contig_ranges:
        for a, b in contig_ranges:
            v.addStyle(
                {'chain': selected_chain, 'resi': f'{a}-{b}'},
                {'cartoon': {'color': 'deepskyblue'}, 'stick': {'radius': 0.2, 'color': 'deepskyblue'}}
            )

    if selected_ligands:
        for token in selected_ligands:
            parsed = parse_ligand_token(token)
            if parsed is None:
                continue
            chain, resn, resi = parsed
            v.addStyle(
                {'chain': chain, 'resn': resn, 'resi': resi, 'hetflag': True},
                {'stick': {'radius': 0.24, 'color': 'magenta'}},
            )

    if hotspot_res.strip():
        for chain, resi in parse_hotspot_tokens(hotspot_res):
            v.addStyle({'chain': chain, 'resi': resi}, {'stick': {'radius': 0.25, 'color': 'red'}})

    v.setHoverable(
        {},
        True,
        "function(atom,viewer,event,container){if(!atom) return; if(atom.__hoverLabel) return; var txt=(atom.resn||'UNK')+' '+(atom.chain||'')+atom.resi; atom.__hoverLabel=viewer.addLabel(txt,{position:atom,backgroundColor:'black',backgroundOpacity:0.65,fontColor:'white',fontSize:12,inFront:true}); viewer.render();}",
        "function(atom,viewer){if(!atom) return; if(atom.__hoverLabel){viewer.removeLabel(atom.__hoverLabel); delete atom.__hoverLabel; viewer.render();}}"
    )
    v.setClickable(
        {},
        True,
        "function(atom,viewer,event,container){if(!atom) return; if(container.__pickedLabel){viewer.removeLabel(container.__pickedLabel); container.__pickedLabel=null;} var txt=(atom.resn||'UNK')+' '+(atom.chain||'')+atom.resi; container.__pickedLabel=viewer.addLabel(txt,{position:atom,backgroundColor:'navy',backgroundOpacity:0.75,fontColor:'white',fontSize:13,inFront:true}); viewer.render();}"
    )
    v.zoomTo()
    return v

source_mode = widgets.Dropdown(
    options=[('Local PDB file', 'local_file'), ('RCSB PDB ID', 'rcsb')],
    value='local_file',
    description='Source:'
)
file_input = widgets.Text(value='', description='File path:', layout=widgets.Layout(width='70%'))
pdbid_input = widgets.Text(value='3BIK', description='PDB ID:', layout=widgets.Layout(width='30%'))
binder_len_input = widgets.Text(value='65-120', description='Binder len:', layout=widgets.Layout(width='30%'))
hotspots_preview = widgets.Text(value='', description='Hotspots:', layout=widgets.Layout(width='38%'))
contigs_preview = widgets.Text(value='', description='Contigs:', layout=widgets.Layout(width='48%'))
ligands_preview = widgets.Text(value='', description='Ligands:', layout=widgets.Layout(width='38%'))
selected_chain_text = widgets.Text(value='', description='Chain:', layout=widgets.Layout(width='25%'))
selection_mode = widgets.ToggleButtons(
    options=[('Select Contigs', 'contigs'), ('Select Hotspots', 'hotspots')],
    value='contigs',
    description='Mode:'
)
ngl_contig_pick_mode = widgets.ToggleButtons(
    options=[('3D Pick: Single', 'single'), ('3D Pick: Range', 'range')],
    value='range',
    description='3D Pick:'
)
show_ligands_chk = widgets.Checkbox(
    value=False,
    description='Show ligands (no waters)',
    indent=False,
    layout=widgets.Layout(width='220px')
)
dragmode_btn = widgets.Button(description='Enable Box Select', tooltip='Sets residue plot drag mode to box select')
clear_anchor_btn = widgets.Button(description='Clear 3D range anchor', tooltip='Clears the current start residue for 3D range picks')
apply_contigs_btn = widgets.Button(description='Apply Contigs Text', tooltip='Parse contigs text and recolor the structure')
clear_contigs_btn = widgets.Button(description='Clear contigs')
clear_hotspots_btn = widgets.Button(description='Clear hotspots')
load_btn = widgets.Button(description='Load + View', button_style='primary')
help_text = widgets.HTML(
    value=(
        '<b>How to select residues:</b> '
        '1) Click <b>Load + View</b>. '
        '2) Optional: check <b>Show ligands (no waters)</b> to display non-standard residues in orange. '
        '3) In <b>Select Contigs</b> mode with <b>nglview</b>, 3D click toggles residues and ligands for contig selection. '
        '4) Optional: switch <b>3D Pick</b> to <b>Range</b>, click start residue then end residue to add a full protein interval. '
        '5) For larger contiguous protein picks, use <b>Enable Box Select</b> in the residue plot.'
    )
)
out = widgets.Output()
viewer_out = widgets.Output()
selector_out = widgets.Output()
chain_btns = widgets.HBox([])
_selector_fig: Optional[go.FigureWidget] = None
_ngl_view: Optional[object] = None

def _extract_ngl_pick(change) -> Optional[Tuple[str, int, str]]:
    picked = change.get('new') if isinstance(change, dict) else None
    if not picked:
        return None
    atom = picked.get('atom1') or {}
    chain = str(atom.get('chainname') or atom.get('chain') or '').strip()
    resi = atom.get('resno') if atom.get('resno') is not None else atom.get('resi')
    resn = str(atom.get('resname') or atom.get('resn') or '').strip().upper()
    if not chain or resi is None:
        return None
    try:
        return chain, int(resi), resn
    except Exception:
        return None

def is_ligand_pick(chain: str, resn: str, resi: int) -> bool:
    if resn in WATER_NAMES:
        return False
    chain_ligs = _state.get('ligand_map', {}).get(chain, [])
    return (resn, resi) in chain_ligs

def on_ngl_pick(change):
    picked = _extract_ngl_pick(change)
    if picked is None:
        return
    chain, resi_i, resn = picked

    if selection_mode.value == 'hotspots':
        token = f'{chain}{resi_i}'
        if token in _state['selected_hotspots']:
            _state['selected_hotspots'].remove(token)
        else:
            _state['selected_hotspots'].add(token)

        refresh_previews()
        refresh_viewer()
        if selected_chain_text.value.strip() == chain:
            make_selector_plot(chain)
        with out:
            print(f'Toggled hotspot from 3D click: {token}')
        return

    chain_id = selected_chain_text.value.strip()
    if not chain_id:
        selected_chain_text.value = chain
        _state['selected_chain'] = chain
        chain_id = chain

    if chain != chain_id:
        with out:
            print(f'3D click ignored: picked chain {chain}, active chain is {chain_id}.')
        return

    if is_ligand_pick(chain, resn, resi_i):
        lig_token = make_ligand_token(chain, resn, resi_i)
        if lig_token in _state['selected_contig_ligands']:
            _state['selected_contig_ligands'].remove(lig_token)
            msg = 'Removed ligand from contig selection'
        else:
            _state['selected_contig_ligands'].add(lig_token)
            msg = 'Added ligand to contig selection'
        _state['ngl_contig_anchor'] = None
        with out:
            print(f'{msg}: {chain}{resi_i}:{resn}')
    else:
        if ngl_contig_pick_mode.value == 'range':
            anchor = _state.get('ngl_contig_anchor')
            if not anchor or anchor[0] != chain:
                _state['ngl_contig_anchor'] = (chain, resi_i)
                with out:
                    print(f'3D range start set at {chain}{resi_i}. Click an end residue to add the range.')
            else:
                a = int(anchor[1])
                b = resi_i
                if a > b:
                    a, b = b, a
                _state['selected_contig_residues'].update(range(a, b + 1))
                _state['ngl_contig_anchor'] = None
                with out:
                    print(f'Added 3D range to contigs: {chain}{a}-{chain}{b}')
        else:
            if resi_i in _state['selected_contig_residues']:
                _state['selected_contig_residues'].remove(resi_i)
            else:
                _state['selected_contig_residues'].add(resi_i)
            _state['ngl_contig_anchor'] = None
            with out:
                print(f'Toggled contig residue from 3D click: {chain}{resi_i}')

    refresh_previews()
    refresh_viewer()
    make_selector_plot(chain_id)

def refresh_viewer():
    global _ngl_view
    with viewer_out:
        clear_output(wait=True)
        if not _state['pdb_text']:
            return
        chain_id = selected_chain_text.value.strip()
        contig_ranges = selected_contigs_as_ranges(_state['selected_contig_residues'])
        hotspot_tokens = parse_hotspot_tokens(hotspots_preview.value.strip())
        anchor = _state.get('ngl_contig_anchor')

        if nv is not None:
            try:
                view = nv.show_text(_state['pdb_text'], ext='pdb')
                view.clear_representations()
                view.add_cartoon(selection='protein', color='lightgray')
                if show_ligands_chk.value:
                    view.add_ball_and_stick(selection='hetero and not water', color='orange')
                if chain_id:
                    view.add_cartoon(selection=f':{chain_id}', color='cornflowerblue')
                if chain_id and contig_ranges:
                    for a, b in contig_ranges:
                        view.add_ball_and_stick(selection=f':{chain_id} and {a}-{b}', color='green')
                for lig_token in _state.get('selected_contig_ligands', set()):
                    parsed = parse_ligand_token(lig_token)
                    if parsed is None:
                        continue
                    l_chain, l_resn, l_resi = parsed
                    view.add_ball_and_stick(selection=f':{l_chain} and {l_resi} and [{l_resn}]', color='magenta')
                if chain_id and anchor and anchor[0] == chain_id:
                    view.add_ball_and_stick(selection=f':{chain_id} and {int(anchor[1])}', color='yellow')
                for h_chain, h_resi in hotspot_tokens:
                    view.add_ball_and_stick(selection=f':{h_chain} and {h_resi}', color='red')
                view.observe(on_ngl_pick, names='picked')
                _ngl_view = view
                display(view)
                return
            except Exception as e:
                print(f'NGL viewer unavailable, using py3Dmol fallback: {e}')

        display(show_viewer_py3dmol(
            _state['pdb_text'],
            chain_id,
            hotspots_preview.value.strip(),
            contig_ranges,
            show_ligands_chk.value,
            _state.get('selected_contig_ligands', set()),
        ).show())

def refresh_previews():
    chain_id = selected_chain_text.value.strip()
    contig_ranges = selected_contigs_as_ranges(_state['selected_contig_residues'])
    contigs_preview.value = format_contigs(chain_id, contig_ranges, binder_len_input.value)
    hotspots_preview.value = format_hotspots(_state['selected_hotspots'])
    ligands_preview.value = format_ligands(_state.get('selected_contig_ligands', set()))

def make_selector_plot(chain_id: str):
    global _selector_fig
    with selector_out:
        clear_output(wait=True)
        residues = _state['residue_map'].get(chain_id, [])
        if not residues:
            print('No residues available for this chain.')
            return

        fig = go.FigureWidget(
            data=[
                go.Scatter(
                    x=residues,
                    y=[0] * len(residues),
                    mode='markers',
                    marker={'size': 9, 'color': ['lightgray'] * len(residues)},
                    text=[f'{chain_id}{resi}' for resi in residues],
                    hovertemplate='Residue %{text}<extra></extra>'
                )
            ]
        )
        fig.update_layout(
            height=240,
            margin={'l': 40, 'r': 20, 't': 30, 'b': 40},
            title=f'Chain {chain_id} residue selector',
            dragmode='select',
            showlegend=False
        )
        fig.update_yaxes(visible=False, fixedrange=True)
        fig.update_xaxes(title='Residue number')

        trace = fig.data[0]

        def redraw_points():
            colors: List[str] = []
            for resi in residues:
                token = f'{chain_id}{resi}'
                if token in _state['selected_hotspots']:
                    colors.append('red')
                elif resi in _state['selected_contig_residues']:
                    colors.append('green')
                else:
                    colors.append('lightgray')
            with fig.batch_update():
                trace.marker.color = colors

        def on_selected(trace_obj, points, selector):
            if selection_mode.value != 'contigs':
                return
            for idx in points.point_inds:
                _state['selected_contig_residues'].add(int(residues[idx]))
            _state['ngl_contig_anchor'] = None
            refresh_previews()
            redraw_points()
            refresh_viewer()

        def on_clicked(trace_obj, points, click_state):
            if not points.point_inds:
                return
            resi = int(residues[points.point_inds[0]])
            token = f'{chain_id}{resi}'
            if selection_mode.value == 'hotspots':
                if token in _state['selected_hotspots']:
                    _state['selected_hotspots'].remove(token)
                else:
                    _state['selected_hotspots'].add(token)
            else:
                if resi in _state['selected_contig_residues']:
                    _state['selected_contig_residues'].remove(resi)
                else:
                    _state['selected_contig_residues'].add(resi)
                _state['ngl_contig_anchor'] = None
            refresh_previews()
            redraw_points()
            refresh_viewer()

        trace.on_selection(on_selected)
        trace.on_click(on_clicked)
        redraw_points()
        _selector_fig = fig
        display(fig)

def enable_box_select(_):
    if _selector_fig is None:
        with out:
            print('Load a model and select a chain first.')
        return
    _selector_fig.update_layout(dragmode='select')

def clear_anchor(_):
    _state['ngl_contig_anchor'] = None
    refresh_viewer()
    with out:
        print('Cleared 3D range anchor.')

def apply_contigs_from_text(_):
    chain_id = selected_chain_text.value.strip()
    if not chain_id:
        with out:
            print('Select a chain first, then apply contigs text.')
        return
    parsed = parse_contigs_for_chain(contigs_preview.value, chain_id)
    _state['selected_contig_residues'] = parsed
    _state['ngl_contig_anchor'] = None
    refresh_previews()
    refresh_viewer()
    make_selector_plot(chain_id)
    with out:
        print(f'Applied contigs for chain {chain_id}: {len(parsed)} residues highlighted.')

def set_chain(chain_id: str):
    selected_chain_text.value = chain_id
    _state['selected_chain'] = chain_id
    _state['selected_contig_residues'] = set()
    _state['selected_contig_ligands'] = set()
    _state['ngl_contig_anchor'] = None
    refresh_previews()
    refresh_viewer()
    make_selector_plot(chain_id)

def build_chain_buttons(chain_ids: List[str]):
    buttons = []
    for chain_id in chain_ids:
        b = widgets.Button(description=f'Chain {chain_id}', layout=widgets.Layout(width='110px'))

        def _on_chain_click(_, cid=chain_id):
            set_chain(cid)

        b.on_click(_on_chain_click)
        buttons.append(b)
    chain_btns.children = tuple(buttons)

def on_load(_):
    global _ngl_view
    with out:
        clear_output(wait=True)
        try:
            pdb_text = load_pdb_text(source_mode.value, file_input.value, pdbid_input.value)
            target_path = OUTDIR / 'inputs' / 'target_original.pdb'
            target_path.write_text(pdb_text)

            chain_ranges, residue_map, ligand_map = parse_chain_data(target_path)
            _state['pdb_text'] = pdb_text
            _state['pdb_path'] = str(target_path)
            _state['chain_ranges'] = chain_ranges
            _state['residue_map'] = residue_map
            _state['ligand_map'] = ligand_map
            _state['selected_hotspots'] = set()
            _state['selected_contig_residues'] = set()
            _state['selected_contig_ligands'] = set()
            _state['ngl_contig_anchor'] = None

            print(f'Saved target PDB: {target_path}')
            print('Detected chain residue ranges:')
            for c, (a, b) in chain_ranges.items():
                print(f'  {c}: {a}-{b}')
            if ligand_map:
                print('Detected non-water ligands:')
                for c in sorted(ligand_map.keys()):
                    tokens = ', '.join([f'{c}{resi}:{resn}' for resn, resi in ligand_map[c]])
                    print(f'  {c}: {tokens}')

            chain_ids = sorted(chain_ranges.keys())
            build_chain_buttons(chain_ids)
            if chain_ids:
                set_chain(chain_ids[0])
            else:
                selected_chain_text.value = ''
                chain_btns.children = tuple()
                with selector_out:
                    clear_output(wait=True)
                    print('No protein chains found in PDB.')
                refresh_viewer()
        except Exception as e:
            print(f'Error: {e}')

def clear_contigs(_):
    _state['selected_contig_residues'] = set()
    _state['selected_contig_ligands'] = set()
    _state['ngl_contig_anchor'] = None
    refresh_previews()
    refresh_viewer()
    if selected_chain_text.value.strip():
        make_selector_plot(selected_chain_text.value.strip())

def clear_hotspots(_):
    _state['selected_hotspots'] = set()
    refresh_previews()
    refresh_viewer()
    if selected_chain_text.value.strip():
        make_selector_plot(selected_chain_text.value.strip())

def on_preview_change(_):
    refresh_viewer()

load_btn.on_click(on_load)
dragmode_btn.on_click(enable_box_select)
clear_anchor_btn.on_click(clear_anchor)
apply_contigs_btn.on_click(apply_contigs_from_text)
clear_contigs_btn.on_click(clear_contigs)
clear_hotspots_btn.on_click(clear_hotspots)
hotspots_preview.observe(on_preview_change, names='value')
show_ligands_chk.observe(on_preview_change, names='value')

viewer_ui = widgets.VBox([
        widgets.HBox([source_mode, file_input, pdbid_input]),
        widgets.HBox([load_btn, selected_chain_text, binder_len_input, show_ligands_chk]),
        help_text,
        chain_btns,
        widgets.HBox([selection_mode, ngl_contig_pick_mode, dragmode_btn, clear_anchor_btn]),
        widgets.HBox([contigs_preview, ligands_preview, hotspots_preview]),
        widgets.HBox([apply_contigs_btn, clear_contigs_btn, clear_hotspots_btn]),
        viewer_out,
        selector_out,
        out,
    ])

In [9]:
# Multi-chain contig selection support
if 'selected_contig_residues_by_chain' not in _state:
    _state['selected_contig_residues_by_chain'] = {}

_chain_button_map: Dict[str, widgets.Button] = {}
refresh_3d_btn = widgets.Button(description='Refresh 3D', tooltip='Refresh 3D viewer now')
auto_refresh_3d_chk = widgets.Checkbox(
    value=True,
    description='Auto-refresh 3D',
    indent=False,
    layout=widgets.Layout(width='170px'),
)


def get_chain_order() -> List[str]:
    return list(_state.get('chain_ranges', {}).keys())


def get_active_chain() -> str:
    return selected_chain_text.value.strip()


def get_selected_residue_map() -> Dict[str, Set[int]]:
    return _state.get('selected_contig_residues_by_chain', {})  # type: ignore[return-value]


def get_selected_residues(chain_id: Optional[str] = None) -> Set[int]:
    residue_map = get_selected_residue_map()
    if chain_id is None:
        selected: Set[int] = set()
        for residues in residue_map.values():
            selected.update(int(resi) for resi in residues)
        return selected
    return set(residue_map.get(chain_id, set()))


def set_selected_residues(chain_id: str, residues: Set[int]):
    chain_id = str(chain_id).strip()
    if not chain_id:
        return
    residue_map = get_selected_residue_map()
    residue_map[chain_id] = set(int(resi) for resi in residues)
    if not residue_map[chain_id]:
        residue_map.pop(chain_id, None)


def toggle_selected_residue(chain_id: str, resi: int):
    chain_id = str(chain_id).strip()
    if not chain_id:
        return
    residue_map = get_selected_residue_map()
    residues = set(residue_map.get(chain_id, set()))
    if int(resi) in residues:
        residues.remove(int(resi))
    else:
        residues.add(int(resi))
    if residues:
        residue_map[chain_id] = residues
    else:
        residue_map.pop(chain_id, None)


def add_selected_residue_range(chain_id: str, start: int, end: int):
    chain_id = str(chain_id).strip()
    if not chain_id:
        return
    if start > end:
        start, end = end, start
    residue_map = get_selected_residue_map()
    residues = set(residue_map.get(chain_id, set()))
    residues.update(range(int(start), int(end) + 1))
    residue_map[chain_id] = residues


def clear_selected_residues():
    _state['selected_contig_residues_by_chain'] = {}


def maybe_refresh_viewer():
    if auto_refresh_3d_chk.value:
        refresh_viewer()


def selected_contigs_by_chain_as_ranges() -> Dict[str, List[Tuple[int, int]]]:
    out: Dict[str, List[Tuple[int, int]]] = {}
    for chain_id, residues in get_selected_residue_map().items():
        if not residues:
            continue
        vals = sorted(int(r) for r in residues)
        ranges: List[Tuple[int, int]] = []
        start = vals[0]
        prev = vals[0]
        for r in vals[1:]:
            if r == prev + 1:
                prev = r
                continue
            ranges.append((start, prev))
            start = r
            prev = r
        ranges.append((start, prev))
        out[chain_id] = ranges
    return out


def format_contigs_from_selection(binder_len: str) -> str:
    chain_ranges = selected_contigs_by_chain_as_ranges()
    if not chain_ranges:
        return ''
    ordered_chain_ids: List[str] = []
    for chain_id in get_chain_order():
        if chain_ranges.get(chain_id):
            ordered_chain_ids.append(chain_id)
    for chain_id in sorted(chain_ranges.keys()):
        if chain_id not in ordered_chain_ids and chain_ranges.get(chain_id):
            ordered_chain_ids.append(chain_id)

    receptor_groups: List[str] = []
    for chain_id in ordered_chain_ids:
        segments = [f'{chain_id}{a}-{b}' for a, b in chain_ranges.get(chain_id, [])]
        if segments:
            receptor_groups.append('/'.join(segments) + '/0')

    receptor = ' '.join(receptor_groups)
    binder = binder_len.strip()
    return f'[{receptor} {binder}]' if binder else f'[{receptor}]'


def parse_contigs_for_all_chains(contig_text: str) -> Dict[str, Set[int]]:
    chain_map: Dict[str, Set[int]] = {}
    body = contig_text.strip().strip('[]').strip()
    if not body:
        return chain_map
    parts = body.split()
    receptor_tokens = parts
    if parts and re.match(r'^\d+-\d+$', parts[-1].strip()):
        receptor_tokens = parts[:-1]
    receptor_part = '/'.join(t.strip() for t in receptor_tokens)
    for frag in receptor_part.split('/'):
        token = frag.strip()
        if not token or token == '0':
            continue
        m = re.match(r'^([A-Za-z])(\d+)(?:-(\d+))?$', token)
        if not m:
            continue
        chain_id = m.group(1)
        start = int(m.group(2))
        end = int(m.group(3)) if m.group(3) else start
        if start > end:
            start, end = end, start
        residues = chain_map.setdefault(chain_id, set())
        residues.update(range(start, end + 1))
    return chain_map


def show_viewer_py3dmol(
    pdb_text: str,
    selected_chain: str = '',
    hotspot_res: str = '',
    contig_ranges: Optional[List[Tuple[int, int]]] = None,
    show_ligands: bool = False,
    selected_ligands: Optional[Set[str]] = None,
):
    v = py3Dmol.view(width=900, height=550)
    v.addModel(pdb_text, 'pdb')

    visible_chains = [chain_id for chain_id in get_chain_order() if chain_id in get_selected_residue_map()]
    if not visible_chains and selected_chain:
        visible_chains = [selected_chain]

    if visible_chains:
        v.setStyle({}, {'cartoon': {'color': 'lightgray'}})
        for idx, chain_id in enumerate(visible_chains):
            chain_style = {'cartoon': {'color': 'cornflowerblue' if idx == 0 else 'spectrum'}}
            v.addStyle({'chain': chain_id}, chain_style)
    else:
        v.setStyle({}, {'cartoon': {'color': 'spectrum'}})

    if show_ligands:
        v.addStyle({'hetflag': True}, {'stick': {'radius': 0.18, 'color': 'orange'}})
        v.setStyle({'hetflag': True, 'resn': ['HOH', 'WAT', 'H2O']}, {})

    if selected_chain and contig_ranges:
        for a, b in contig_ranges:
            v.addStyle(
                {'chain': selected_chain, 'resi': f'{a}-{b}'},
                {'cartoon': {'color': 'deepskyblue'}, 'stick': {'radius': 0.2, 'color': 'deepskyblue'}}
            )

    if selected_ligands:
        for token in selected_ligands:
            parsed = parse_ligand_token(token)
            if parsed is None:
                continue
            chain, resn, resi = parsed
            v.addStyle(
                {'chain': chain, 'resn': resn, 'resi': resi, 'hetflag': True},
                {'stick': {'radius': 0.24, 'color': 'magenta'}},
            )

    if hotspot_res.strip():
        for chain, resi in parse_hotspot_tokens(hotspot_res):
            v.addStyle({'chain': chain, 'resi': resi}, {'stick': {'radius': 0.25, 'color': 'red'}})

    v.setHoverable(
        {},
        True,
        "function(atom,viewer,event,container){if(!atom) return; if(atom.__hoverLabel) return; var txt=(atom.resn||'UNK')+' '+(atom.chain||'')+atom.resi; atom.__hoverLabel=viewer.addLabel(txt,{position:atom,backgroundColor:'black',backgroundOpacity:0.65,fontColor:'white',fontSize:12,inFront:true}); viewer.render();}",
        "function(atom,viewer){if(!atom) return; if(atom.__hoverLabel){viewer.removeLabel(atom.__hoverLabel); delete atom.__hoverLabel; viewer.render();}}"
    )
    v.setClickable(
        {},
        True,
        "function(atom,viewer,event,container){if(!atom) return; if(container.__pickedLabel){viewer.removeLabel(container.__pickedLabel); container.__pickedLabel=null;} var txt=(atom.resn||'UNK')+' '+(atom.chain||'')+atom.resi; container.__pickedLabel=viewer.addLabel(txt,{position:atom,backgroundColor:'navy',backgroundOpacity:0.75,fontColor:'white',fontSize:13,inFront:true}); viewer.render();}"
    )
    v.zoomTo()
    return v


def _apply_ngl_representations(view, chain_ids: List[str], active_chain: str, hotspot_tokens, anchor):
    view.clear_representations()
    view.add_cartoon(selection='protein', color='lightgray')
    for idx, chain_id in enumerate(chain_ids):
        view.add_cartoon(selection=f':{chain_id}', color='cornflowerblue' if idx == 0 else 'spectrum')
    if show_ligands_chk.value:
        view.add_ball_and_stick(selection='hetero and not water', color='orange')
    for chain_id, ranges in selected_contigs_by_chain_as_ranges().items():
        for a, b in ranges:
            view.add_ball_and_stick(selection=f':{chain_id} and {a}-{b}', color='green')
    for lig_token in _state.get('selected_contig_ligands', set()):
        parsed = parse_ligand_token(lig_token)
        if parsed is None:
            continue
        l_chain, l_resn, l_resi = parsed
        view.add_ball_and_stick(selection=f':{l_chain} and {l_resi} and [{l_resn}]', color='magenta')
    if active_chain and anchor and anchor[0] == active_chain:
        view.add_ball_and_stick(selection=f':{active_chain} and {int(anchor[1])}', color='yellow')
    for h_chain, h_resi in hotspot_tokens:
        view.add_ball_and_stick(selection=f':{h_chain} and {h_resi}', color='red')


def refresh_viewer():
    global _ngl_view
    if not _state['pdb_text']:
        return

    active_chain = get_active_chain()
    chain_ids = list(selected_contigs_by_chain_as_ranges().keys())
    if not chain_ids and active_chain:
        chain_ids = [active_chain]
    contig_ranges = selected_contigs_by_chain_as_ranges().get(active_chain, [])
    hotspot_tokens = parse_hotspot_tokens(hotspots_preview.value.strip())
    anchor = _state.get('ngl_contig_anchor')

    if nv is not None:
        try:
            if _ngl_view is None:
                view = nv.show_text(_state['pdb_text'], ext='pdb')
                view.observe(on_ngl_pick, names='picked')
                _ngl_view = view
                with viewer_out:
                    clear_output(wait=True)
                    display(view)
            _apply_ngl_representations(_ngl_view, chain_ids, active_chain, hotspot_tokens, anchor)
            return
        except Exception as e:
            _ngl_view = None
            with viewer_out:
                clear_output(wait=True)
                print(f'NGL viewer unavailable, using py3Dmol fallback: {e}')

    with viewer_out:
        clear_output(wait=True)
        display(show_viewer_py3dmol(
            _state['pdb_text'],
            active_chain,
            hotspots_preview.value.strip(),
            contig_ranges,
            show_ligands_chk.value,
            _state.get('selected_contig_ligands', set()),
        ).show())


def refresh_previews():
    contigs_preview.value = format_contigs_from_selection(binder_len_input.value)
    hotspots_preview.value = format_hotspots(_state['selected_hotspots'])
    ligands_preview.value = format_ligands(_state.get('selected_contig_ligands', set()))


def make_selector_plot(chain_id: str):
    global _selector_fig
    with selector_out:
        clear_output(wait=True)
        residues = _state['residue_map'].get(chain_id, [])
        if not residues:
            print('No residues available for this chain.')
            return

        fig = go.FigureWidget(
            data=[
                go.Scatter(
                    x=residues,
                    y=[0] * len(residues),
                    mode='markers',
                    marker={'size': 9, 'color': ['lightgray'] * len(residues)},
                    text=[f'{chain_id}{resi}' for resi in residues],
                    hovertemplate='Residue %{text}<extra></extra>'
                )
            ]
        )
        fig.update_layout(
            height=240,
            margin={'l': 40, 'r': 20, 't': 30, 'b': 40},
            title=f'Chain {chain_id} residue selector',
            dragmode='select',
            showlegend=False
        )
        fig.update_yaxes(visible=False, fixedrange=True)
        fig.update_xaxes(title='Residue number')

        trace = fig.data[0]
        selected_residues = set(get_selected_residues(chain_id))

        def redraw_points():
            colors: List[str] = []
            selected_residues_local = get_selected_residues(chain_id)
            for resi in residues:
                token = f'{chain_id}{resi}'
                if token in _state['selected_hotspots']:
                    colors.append('red')
                elif resi in selected_residues_local:
                    colors.append('green')
                else:
                    colors.append('lightgray')
            with fig.batch_update():
                trace.marker.color = colors

        def on_selected(trace_obj, points, selector):
            if selection_mode.value != 'contigs':
                return
            if not points.point_inds:
                return
            picked_residues = [int(residues[idx]) for idx in points.point_inds]
            if ngl_contig_pick_mode.value == 'range':
                add_selected_residue_range(chain_id, min(picked_residues), max(picked_residues))
            else:
                for resi in picked_residues:
                    set_selected = get_selected_residues(chain_id)
                    set_selected.add(resi)
                    set_selected_residues(chain_id, set_selected)
            _state['ngl_contig_anchor'] = None
            refresh_previews()
            redraw_points()
            maybe_refresh_viewer()

        def on_clicked(trace_obj, points, click_state):
            if not points.point_inds:
                return
            resi = int(residues[points.point_inds[0]])
            token = f'{chain_id}{resi}'
            if selection_mode.value == 'hotspots':
                if token in _state['selected_hotspots']:
                    _state['selected_hotspots'].remove(token)
                else:
                    _state['selected_hotspots'].add(token)
            else:
                if ngl_contig_pick_mode.value == 'range':
                    anchor = _state.get('ngl_contig_anchor')
                    if not anchor or anchor[0] != chain_id:
                        _state['ngl_contig_anchor'] = (chain_id, resi)
                    else:
                        add_selected_residue_range(chain_id, int(anchor[1]), resi)
                        _state['ngl_contig_anchor'] = None
                else:
                    toggle_selected_residue(chain_id, resi)
                    _state['ngl_contig_anchor'] = None
            refresh_previews()
            redraw_points()
            maybe_refresh_viewer()

        trace.on_selection(on_selected)
        trace.on_click(on_clicked)
        redraw_points()
        _selector_fig = fig
        display(fig)


def enable_box_select(_):
    if _selector_fig is None:
        with out:
            print('Load a model and select a chain first.')
        return
    _selector_fig.update_layout(dragmode='select')


def clear_anchor(_):
    _state['ngl_contig_anchor'] = None
    refresh_viewer()
    with out:
        print('Cleared 3D range anchor.')


def apply_contigs_from_text(_):
    contig_map = parse_contigs_for_all_chains(contigs_preview.value)
    if not contig_map:
        with out:
            print('No contigs found in contigs text.')
        return
    _state['selected_contig_residues_by_chain'] = contig_map
    _state['ngl_contig_anchor'] = None
    first_chain = next(iter(contig_map.keys()))
    if first_chain:
        selected_chain_text.value = first_chain
    refresh_previews()
    refresh_viewer()
    make_selector_plot(get_active_chain() or first_chain)
    with out:
        print(f'Applied contigs across {len(contig_map)} chain(s).')


def set_chain(chain_id: str):
    chain_id = str(chain_id).strip()
    if not chain_id:
        return
    selected_chain_text.value = chain_id
    sync_chain_button_styles()
    refresh_previews()
    refresh_viewer()
    make_selector_plot(chain_id)


def sync_chain_button_styles():
    active_chain = get_active_chain()
    for chain_id, button in _chain_button_map.items():
        button.button_style = 'success' if chain_id == active_chain else ''


def build_chain_buttons(chain_ids: List[str]):
    buttons = []
    _chain_button_map.clear()
    for chain_id in chain_ids:
        button = widgets.Button(description=f'Chain {chain_id}', layout=widgets.Layout(width='110px'))

        def _on_chain_click(_, cid=chain_id):
            set_chain(cid)

        button.on_click(_on_chain_click)
        buttons.append(button)
        _chain_button_map[chain_id] = button
    chain_btns.children = tuple(buttons)
    sync_chain_button_styles()


def on_ngl_pick(change):
    picked = _extract_ngl_pick(change)
    if picked is None:
        return
    chain, resi_i, resn = picked

    if selection_mode.value == 'hotspots':
        token = f'{chain}{resi_i}'
        if token in _state['selected_hotspots']:
            _state['selected_hotspots'].remove(token)
        else:
            _state['selected_hotspots'].add(token)
        if chain != get_active_chain():
            selected_chain_text.value = chain
            sync_chain_button_styles()
        refresh_previews()
        maybe_refresh_viewer()
        if get_active_chain() == chain:
            make_selector_plot(chain)
        with out:
            print(f'Toggled hotspot from 3D click: {token}')
        return

    if chain != get_active_chain():
        selected_chain_text.value = chain
        sync_chain_button_styles()

    if is_ligand_pick(chain, resn, resi_i):
        lig_token = make_ligand_token(chain, resn, resi_i)
        if lig_token in _state['selected_contig_ligands']:
            _state['selected_contig_ligands'].remove(lig_token)
            msg = 'Removed ligand from contig selection'
        else:
            _state['selected_contig_ligands'].add(lig_token)
            msg = 'Added ligand to contig selection'
        _state['ngl_contig_anchor'] = None
        with out:
            print(f'{msg}: {chain}{resi_i}:{resn}')
    else:
        if ngl_contig_pick_mode.value == 'range':
            anchor = _state.get('ngl_contig_anchor')
            if not anchor or anchor[0] != chain:
                _state['ngl_contig_anchor'] = (chain, resi_i)
                with out:
                    print(f'3D range start set at {chain}{resi_i}. Click an end residue to add the range.')
            else:
                a = int(anchor[1])
                b = resi_i
                add_selected_residue_range(chain, a, b)
                _state['ngl_contig_anchor'] = None
                with out:
                    print(f'Added 3D range to contigs: {chain}{min(a, b)}-{chain}{max(a, b)}')
        else:
            toggle_selected_residue(chain, resi_i)
            _state['ngl_contig_anchor'] = None
            with out:
                print(f'Toggled contig residue from 3D click: {chain}{resi_i}')

    refresh_previews()
    maybe_refresh_viewer()
    make_selector_plot(chain)


def on_load(_):
    with out:
        clear_output(wait=True)
        try:
            pdb_text = load_pdb_text(source_mode.value, file_input.value, pdbid_input.value)
            target_path = OUTDIR / 'inputs' / 'target_original.pdb'
            target_path.write_text(pdb_text)

            chain_ranges, residue_map, ligand_map = parse_chain_data(target_path)
            _state['pdb_text'] = pdb_text
            _state['pdb_path'] = str(target_path)
            _state['chain_ranges'] = chain_ranges
            _state['residue_map'] = residue_map
            _state['ligand_map'] = ligand_map
            _state['selected_hotspots'] = set()
            _state['selected_contig_ligands'] = set()
            _state['ngl_contig_anchor'] = None
            _ngl_view = None
            clear_selected_residues()

            print(f'Saved target PDB: {target_path}')
            print('Detected chain residue ranges:')
            for c, (a, b) in chain_ranges.items():
                print(f'  {c}: {a}-{b}')
            if ligand_map:
                print('Detected non-water ligands:')
                for c in sorted(ligand_map.keys()):
                    tokens = ', '.join([f'{c}{resi}:{resn}' for resn, resi in ligand_map[c]])
                    print(f'  {c}: {tokens}')

            chain_ids = sorted(chain_ranges.keys())
            build_chain_buttons(chain_ids)
            if chain_ids:
                set_chain(chain_ids[0])
            else:
                selected_chain_text.value = ''
                chain_btns.children = tuple()
                with selector_out:
                    clear_output(wait=True)
                    print('No protein chains found in PDB.')
                refresh_viewer()
        except Exception as e:
            print(f'Error: {e}')


def clear_contigs(_):
    clear_selected_residues()
    _state['selected_contig_ligands'] = set()
    _state['ngl_contig_anchor'] = None
    refresh_previews()
    refresh_viewer()
    active_chain = get_active_chain()
    if active_chain:
        make_selector_plot(active_chain)


def clear_hotspots(_):
    _state['selected_hotspots'] = set()
    refresh_previews()
    refresh_viewer()
    active_chain = get_active_chain()
    if active_chain:
        make_selector_plot(active_chain)


def on_preview_change(_):
    refresh_viewer()


def manual_refresh_viewer(_):
    refresh_viewer()


def _reset_button_click_handlers(button):
    try:
        button._click_handlers.callbacks = []
    except Exception:
        pass


for _button in (load_btn, dragmode_btn, clear_anchor_btn, apply_contigs_btn, clear_contigs_btn, clear_hotspots_btn, refresh_3d_btn):
    _reset_button_click_handlers(_button)


load_btn.on_click(on_load)
dragmode_btn.on_click(enable_box_select)
clear_anchor_btn.on_click(clear_anchor)
apply_contigs_btn.on_click(apply_contigs_from_text)
clear_contigs_btn.on_click(clear_contigs)
clear_hotspots_btn.on_click(clear_hotspots)
refresh_3d_btn.on_click(manual_refresh_viewer)
hotspots_preview.observe(on_preview_change, names='value')
show_ligands_chk.observe(on_preview_change, names='value')

viewer_ui = widgets.VBox([
        widgets.HBox([source_mode, file_input, pdbid_input]),
        widgets.HBox([load_btn, selected_chain_text, binder_len_input, show_ligands_chk]),
        help_text,
        chain_btns,
        widgets.HBox([selection_mode, ngl_contig_pick_mode, dragmode_btn, clear_anchor_btn, refresh_3d_btn, auto_refresh_3d_chk]),
        widgets.HBox([contigs_preview, ligands_preview, hotspots_preview]),
        widgets.HBox([apply_contigs_btn, clear_contigs_btn, clear_hotspots_btn]),
        viewer_out,
        selector_out,
        out,
    ])


In [10]:
# CIF/PDB compatibility patch: auto-fetch best available format and normalize downstream IDs.
import io
import tempfile
from Bio.PDB import MMCIFParser, PDBIO

_state.setdefault('structure_format', 'pdb')
_state.setdefault('viewer_format', 'pdb')
_state.setdefault('structure_source', '')


def _parse_structure_text(structure_text: str, structure_format: str):
    fmt = str(structure_format).strip().lower()
    if fmt not in {'pdb', 'cif'}:
        raise ValueError(f'Unsupported structure format: {structure_format}')
    parser = PDBParser(QUIET=True) if fmt == 'pdb' else MMCIFParser(QUIET=True)
    suffix = '.pdb' if fmt == 'pdb' else '.cif'
    with tempfile.NamedTemporaryFile('w', suffix=suffix, delete=False) as tmp:
        tmp.write(structure_text)
        tmp_path = Path(tmp.name)
    try:
        structure = parser.get_structure('target', str(tmp_path))
    finally:
        try:
            tmp_path.unlink(missing_ok=True)
        except Exception:
            pass
    return structure


def parse_chain_data_from_text(
    structure_text: str,
    structure_format: str,
) -> Tuple[Dict[str, Tuple[int, int]], Dict[str, List[int]], Dict[str, List[Tuple[str, int]]]]:
    structure = _parse_structure_text(structure_text, structure_format)
    chain_ranges: Dict[str, Tuple[int, int]] = {}
    residue_map: Dict[str, List[int]] = {}
    ligand_map: Dict[str, List[Tuple[str, int]]] = {}

    for model in structure:
        for chain in model:
            chain_id = str(chain.id).strip()
            residues = sorted({int(r.get_id()[1]) for r in chain.get_residues() if r.get_id()[0] == ' '})
            if residues:
                residue_map[chain_id] = residues
                chain_ranges[chain_id] = (min(residues), max(residues))

            ligands: List[Tuple[str, int]] = []
            for r in chain.get_residues():
                hetflag, resseq, _icode = r.get_id()
                if hetflag == ' ':
                    continue
                resn = str(r.get_resname()).strip().upper()
                if resn in WATER_NAMES:
                    continue
                try:
                    ligands.append((resn, int(resseq)))
                except Exception:
                    continue
            if ligands:
                ligand_map[chain_id] = sorted(set(ligands), key=lambda x: (x[1], x[0]))
        break

    return chain_ranges, residue_map, ligand_map


def _write_structure_as_pdb(structure_text: str, structure_format: str, out_path: Path):
    if structure_format == 'pdb':
        out_path.write_text(structure_text)
        return
    structure = _parse_structure_text(structure_text, structure_format)
    io = PDBIO()
    io.set_structure(structure)
    io.save(str(out_path))


def _structure_text_to_pdb_text(structure_text: str, structure_format: str) -> str:
    if structure_format == 'pdb':
        return structure_text
    structure = _parse_structure_text(structure_text, structure_format)
    io_obj = PDBIO()
    io_obj.set_structure(structure)
    handle = io.StringIO()
    io_obj.save(handle)
    return handle.getvalue()


def fetch_structure_from_rcsb(pdb_id: str) -> Tuple[str, str, str]:
    pdb_id = pdb_id.strip().upper()
    if not re.match(r'^[A-Z0-9]{4}$', pdb_id):
        raise ValueError('PDB ID should be 4 alphanumeric characters.')

    tried: List[str] = []
    for ext in ('pdb', 'cif'):
        url = f'https://files.rcsb.org/download/{pdb_id}.{ext}'
        tried.append(url)
        r = requests.get(url, timeout=30)
        if r.ok and r.text.strip():
            return r.text, ext, url

    raise ValueError(
        f'Could not download {pdb_id} from RCSB as .pdb or .cif. Tried: ' + ', '.join(tried)
    )


def load_structure_text(source_mode: str, file_path: str, pdb_id: str) -> Tuple[str, str, str]:
    if source_mode == 'local_file':
        if not file_path.strip():
            raise ValueError('Please enter a PDB/CIF file path, or switch Source to RCSB PDB ID.')
        p = Path(file_path).expanduser().resolve()
        if not p.exists():
            raise FileNotFoundError(f'File not found: {p}')
        if p.is_dir():
            raise IsADirectoryError(f'Path is a directory, not a structure file: {p}')
        if not p.is_file():
            raise ValueError(f'Path is not a regular file: {p}')

        suffix = p.suffix.strip().lower()
        if suffix in {'.pdb', '.ent'}:
            fmt = 'pdb'
        elif suffix in {'.cif', '.mmcif'}:
            fmt = 'cif'
        else:
            raise ValueError('Local file must end with .pdb, .ent, .cif, or .mmcif.')
        return p.read_text(), fmt, str(p)

    return fetch_structure_from_rcsb(pdb_id)


def load_pdb_text(source_mode: str, file_path: str, pdb_id: str) -> str:
    # Backward-compatible helper expected by earlier cells.
    text, fmt, source = load_structure_text(source_mode, file_path, pdb_id)
    _state['structure_format'] = fmt
    _state['structure_source'] = source
    return text


def parse_chain_data(pdb_path: Path):
    fmt = str(_state.get('structure_format', 'pdb')).strip().lower()
    structure_text = Path(pdb_path).read_text()
    return parse_chain_data_from_text(structure_text, fmt)


def parse_hotspot_token(token: str) -> Optional[Tuple[str, int]]:
    t = token.strip()
    if not t:
        return None

    m = re.match(r'^([A-Za-z0-9]+):(-?\d+)$', t)
    if m:
        return m.group(1), int(m.group(2))

    m = re.match(r'^([A-Za-z])(-?\d+)$', t)
    if m:
        return m.group(1), int(m.group(2))

    return None


def make_hotspot_token(chain: str, resi: int) -> str:
    return f'{chain}:{int(resi)}'


def parse_hotspot_tokens(hotspot_res: str) -> List[Tuple[str, int]]:
    out: List[Tuple[str, int]] = []
    for raw in [x.strip() for x in hotspot_res.split(',') if x.strip()]:
        parsed = parse_hotspot_token(raw)
        if parsed is not None:
            out.append(parsed)
    return out


def format_hotspots(tokens: Set[str]) -> str:
    parsed = [p for p in (parse_hotspot_token(t) for t in tokens) if p is not None]
    ordered = sorted(parsed, key=lambda x: (x[0], x[1]))
    normalized = [f'{c}{r}' if len(c) == 1 else f'{c}:{r}' for c, r in ordered]
    return ','.join(normalized)


def parse_contigs_for_all_chains(contig_text: str) -> Dict[str, Set[int]]:
    chain_map: Dict[str, Set[int]] = {}
    body = contig_text.strip().strip('[]').strip()
    if not body:
        return chain_map

    parts = body.split()
    receptor_tokens = parts
    if parts and re.match(r'^-?\d+-\d+$', parts[-1].strip()):
        receptor_tokens = parts[:-1]

    receptor_part = '/'.join(t.strip() for t in receptor_tokens)
    known_chain_ids = sorted(
        [str(c).strip() for c in _state.get('chain_ranges', {}).keys() if str(c).strip()],
        key=len,
        reverse=True,
    )

    for frag in receptor_part.split('/'):
        token = frag.strip()
        if not token or token == '0':
            continue

        chain_id = ''
        range_part = ''
        for cid in known_chain_ids:
            if token.startswith(cid):
                candidate = token[len(cid):]
                if re.match(r'^-?\d+(?:--?\d+)?$', candidate):
                    chain_id = cid
                    range_part = candidate
                    break

        if not chain_id:
            m = re.match(r'^([A-Za-z])(-?\d+)(?:-(-?\d+))?$', token)
            if not m:
                continue
            chain_id = m.group(1)
            start = int(m.group(2))
            end = int(m.group(3)) if m.group(3) else start
        else:
            m = re.match(r'^(-?\d+)(?:-(-?\d+))?$', range_part)
            if not m:
                continue
            start = int(m.group(1))
            end = int(m.group(2)) if m.group(2) else start

        if start > end:
            start, end = end, start
        residues = chain_map.setdefault(chain_id, set())
        residues.update(range(start, end + 1))

    return chain_map


def show_viewer_py3dmol(
    pdb_text: str,
    selected_chain: str = '',
    hotspot_res: str = '',
    contig_ranges: Optional[List[Tuple[int, int]]] = None,
    show_ligands: bool = False,
    selected_ligands: Optional[Set[str]] = None,
):
    v = py3Dmol.view(width=900, height=550)
    structure_format = str(_state.get('viewer_format', _state.get('structure_format', 'pdb'))).strip().lower()
    v.addModel(pdb_text, structure_format)

    visible_chains = [chain_id for chain_id in get_chain_order() if chain_id in get_selected_residue_map()]
    if not visible_chains and selected_chain:
        visible_chains = [selected_chain]

    if visible_chains:
        v.setStyle({}, {'cartoon': {'color': 'lightgray'}})
        for idx, chain_id in enumerate(visible_chains):
            chain_style = {'cartoon': {'color': 'cornflowerblue' if idx == 0 else 'spectrum'}}
            v.addStyle({'chain': chain_id}, chain_style)
    else:
        v.setStyle({}, {'cartoon': {'color': 'spectrum'}})

    if show_ligands:
        v.addStyle({'hetflag': True}, {'stick': {'radius': 0.18, 'color': 'orange'}})
        v.setStyle({'hetflag': True, 'resn': ['HOH', 'WAT', 'H2O']}, {})

    if selected_chain and contig_ranges:
        for a, b in contig_ranges:
            v.addStyle(
                {'chain': selected_chain, 'resi': f'{a}-{b}'},
                {'cartoon': {'color': 'deepskyblue'}, 'stick': {'radius': 0.2, 'color': 'deepskyblue'}},
            )

    if selected_ligands:
        for token in selected_ligands:
            parsed = parse_ligand_token(token)
            if parsed is None:
                continue
            chain, resn, resi = parsed
            v.addStyle(
                {'chain': chain, 'resn': resn, 'resi': resi, 'hetflag': True},
                {'stick': {'radius': 0.24, 'color': 'magenta'}},
            )

    if hotspot_res.strip():
        for chain, resi in parse_hotspot_tokens(hotspot_res):
            v.addStyle({'chain': chain, 'resi': resi}, {'stick': {'radius': 0.25, 'color': 'red'}})

    v.setHoverable(
        {},
        True,
        "function(atom,viewer,event,container){if(!atom) return; if(atom.__hoverLabel) return; var txt=(atom.resn||'UNK')+' '+(atom.chain||'')+atom.resi; atom.__hoverLabel=viewer.addLabel(txt,{position:atom,backgroundColor:'black',backgroundOpacity:0.65,fontColor:'white',fontSize:12,inFront:true}); viewer.render();}",
        "function(atom,viewer){if(!atom) return; if(atom.__hoverLabel){viewer.removeLabel(atom.__hoverLabel); delete atom.__hoverLabel; viewer.render();}}",
    )
    v.setClickable(
        {},
        True,
        "function(atom,viewer,event,container){if(!atom) return; if(container.__pickedLabel){viewer.removeLabel(container.__pickedLabel); container.__pickedLabel=null;} var txt=(atom.resn||'UNK')+' '+(atom.chain||'')+atom.resi; container.__pickedLabel=viewer.addLabel(txt,{position:atom,backgroundColor:'navy',backgroundOpacity:0.75,fontColor:'white',fontSize:13,inFront:true}); viewer.render();}",
    )
    v.zoomTo()
    return v


def refresh_viewer():
    global _ngl_view
    if not _state['pdb_text']:
        return

    active_chain = get_active_chain()
    chain_ids = list(selected_contigs_by_chain_as_ranges().keys())
    if not chain_ids and active_chain:
        chain_ids = [active_chain]
    contig_ranges = selected_contigs_by_chain_as_ranges().get(active_chain, [])
    hotspot_tokens = parse_hotspot_tokens(hotspots_preview.value.strip())
    anchor = _state.get('ngl_contig_anchor')
    structure_ext = str(_state.get('viewer_format', _state.get('structure_format', 'pdb'))).strip().lower()

    if nv is not None:
        try:
            if _ngl_view is None:
                view = nv.show_text(_state['pdb_text'], ext=structure_ext)
                view.observe(on_ngl_pick, names='picked')
                _ngl_view = view
                with viewer_out:
                    clear_output(wait=True)
                    display(view)
            _apply_ngl_representations(_ngl_view, chain_ids, active_chain, hotspot_tokens, anchor)
            return
        except Exception as e:
            _ngl_view = None
            with viewer_out:
                clear_output(wait=True)
                print(f'NGL viewer unavailable, using py3Dmol fallback: {e}')

    with viewer_out:
        clear_output(wait=True)
        display(show_viewer_py3dmol(
            _state['pdb_text'],
            active_chain,
            hotspots_preview.value.strip(),
            contig_ranges,
            show_ligands_chk.value,
            _state.get('selected_contig_ligands', set()),
        ).show())


def on_ngl_pick(change):
    picked = _extract_ngl_pick(change)
    if picked is None:
        return
    chain, resi_i, resn = picked

    if selection_mode.value == 'hotspots':
        token = make_hotspot_token(chain, resi_i)
        if token in _state['selected_hotspots']:
            _state['selected_hotspots'].remove(token)
        else:
            _state['selected_hotspots'].add(token)
        if chain != get_active_chain():
            selected_chain_text.value = chain
            sync_chain_button_styles()
        refresh_previews()
        maybe_refresh_viewer()
        if get_active_chain() == chain:
            make_selector_plot(chain)
        with out:
            print(f'Toggled hotspot from 3D click: {chain}:{resi_i}')
        return

    if chain != get_active_chain():
        selected_chain_text.value = chain
        sync_chain_button_styles()

    if is_ligand_pick(chain, resn, resi_i):
        lig_token = make_ligand_token(chain, resn, resi_i)
        if lig_token in _state['selected_contig_ligands']:
            _state['selected_contig_ligands'].remove(lig_token)
            msg = 'Removed ligand from contig selection'
        else:
            _state['selected_contig_ligands'].add(lig_token)
            msg = 'Added ligand to contig selection'
        _state['ngl_contig_anchor'] = None
        with out:
            print(f'{msg}: {chain}{resi_i}:{resn}')
    else:
        if ngl_contig_pick_mode.value == 'range':
            anchor = _state.get('ngl_contig_anchor')
            if not anchor or anchor[0] != chain:
                _state['ngl_contig_anchor'] = (chain, resi_i)
                with out:
                    print(f'3D range start set at {chain}{resi_i}. Click an end residue to add the range.')
            else:
                a = int(anchor[1])
                b = resi_i
                add_selected_residue_range(chain, a, b)
                _state['ngl_contig_anchor'] = None
                with out:
                    print(f'Added 3D range to contigs: {chain}{min(a, b)}-{chain}{max(a, b)}')
        else:
            toggle_selected_residue(chain, resi_i)
            _state['ngl_contig_anchor'] = None
            with out:
                print(f'Toggled contig residue from 3D click: {chain}{resi_i}')

    refresh_previews()
    maybe_refresh_viewer()
    make_selector_plot(chain)


def make_selector_plot(chain_id: str):
    global _selector_fig
    with selector_out:
        clear_output(wait=True)
        residues = _state['residue_map'].get(chain_id, [])
        if not residues:
            print('No residues available for this chain.')
            return

        fig = go.FigureWidget(
            data=[
                go.Scatter(
                    x=residues,
                    y=[0] * len(residues),
                    mode='markers',
                    marker={'size': 9, 'color': ['lightgray'] * len(residues)},
                    text=[f'{chain_id}{resi}' for resi in residues],
                    hovertemplate='Residue %{text}<extra></extra>',
                )
            ]
        )
        fig.update_layout(
            height=240,
            margin={'l': 40, 'r': 20, 't': 30, 'b': 40},
            title=f'Chain {chain_id} residue selector',
            dragmode='select',
            showlegend=False,
        )
        fig.update_yaxes(visible=False, fixedrange=True)
        fig.update_xaxes(title='Residue number')

        trace = fig.data[0]

        def redraw_points():
            colors: List[str] = []
            selected_residues_local = get_selected_residues(chain_id)
            for resi in residues:
                token = make_hotspot_token(chain_id, int(resi))
                if token in _state['selected_hotspots']:
                    colors.append('red')
                elif resi in selected_residues_local:
                    colors.append('green')
                else:
                    colors.append('lightgray')
            with fig.batch_update():
                trace.marker.color = colors

        def on_selected(trace_obj, points, selector):
            if selection_mode.value != 'contigs':
                return
            if not points.point_inds:
                return
            picked_residues = [int(residues[idx]) for idx in points.point_inds]
            if ngl_contig_pick_mode.value == 'range':
                add_selected_residue_range(chain_id, min(picked_residues), max(picked_residues))
            else:
                for resi in picked_residues:
                    set_selected = get_selected_residues(chain_id)
                    set_selected.add(resi)
                    set_selected_residues(chain_id, set_selected)
            _state['ngl_contig_anchor'] = None
            refresh_previews()
            redraw_points()
            maybe_refresh_viewer()

        def on_clicked(trace_obj, points, click_state):
            if not points.point_inds:
                return
            resi = int(residues[points.point_inds[0]])
            token = make_hotspot_token(chain_id, resi)
            if selection_mode.value == 'hotspots':
                if token in _state['selected_hotspots']:
                    _state['selected_hotspots'].remove(token)
                else:
                    _state['selected_hotspots'].add(token)
            else:
                if ngl_contig_pick_mode.value == 'range':
                    anchor = _state.get('ngl_contig_anchor')
                    if not anchor or anchor[0] != chain_id:
                        _state['ngl_contig_anchor'] = (chain_id, resi)
                    else:
                        add_selected_residue_range(chain_id, int(anchor[1]), resi)
                        _state['ngl_contig_anchor'] = None
                else:
                    toggle_selected_residue(chain_id, resi)
                    _state['ngl_contig_anchor'] = None
            refresh_previews()
            redraw_points()
            maybe_refresh_viewer()

        trace.on_selection(on_selected)
        trace.on_click(on_clicked)
        redraw_points()
        _selector_fig = fig
        display(fig)


def on_load(_):
    global _ngl_view
    with out:
        clear_output(wait=True)
        try:
            structure_text, structure_format, source_path = load_structure_text(
                source_mode.value, file_input.value, pdbid_input.value
            )

            raw_target_path = OUTDIR / 'inputs' / f'target_original.{structure_format}'
            raw_target_path.write_text(structure_text)

            target_pdb_path = OUTDIR / 'inputs' / 'target_original.pdb'
            _write_structure_as_pdb(structure_text, structure_format, target_pdb_path)

            chain_ranges, residue_map, ligand_map = parse_chain_data_from_text(structure_text, structure_format)
            viewer_text = structure_text
            viewer_format = structure_format
            if structure_format == 'cif':
                try:
                    viewer_text = _structure_text_to_pdb_text(structure_text, structure_format)
                    viewer_format = 'pdb'
                except Exception as conv_err:
                    viewer_text = structure_text
                    viewer_format = 'cif'
                    print(f'Warning: CIF-to-PDB viewer conversion failed; using CIF directly ({conv_err})')

            _state['pdb_text'] = viewer_text
            _state['pdb_path'] = str(target_pdb_path)
            _state['structure_format'] = structure_format
            _state['viewer_format'] = viewer_format
            _state['structure_source'] = source_path
            _state['chain_ranges'] = chain_ranges
            _state['residue_map'] = residue_map
            _state['ligand_map'] = ligand_map
            _state['selected_hotspots'] = set()
            _state['selected_contig_ligands'] = set()
            _state['ngl_contig_anchor'] = None
            _ngl_view = None
            clear_selected_residues()

            print(f'Loaded structure format: {structure_format.upper()}')
            print(f'Viewer format: {viewer_format.upper()}')
            print(f'Source: {source_path}')
            print(f'Saved original structure: {raw_target_path}')
            if structure_format == 'cif':
                print(f'Generated downstream PDB: {target_pdb_path}')
            else:
                print(f'Saved target PDB: {target_pdb_path}')
            print('Detected chain residue ranges:')
            for c, (a, b) in chain_ranges.items():
                print(f'  {c}: {a}-{b}')
            if ligand_map:
                print('Detected non-water ligands:')
                for c in sorted(ligand_map.keys()):
                    tokens = ', '.join([f'{c}{resi}:{resn}' for resn, resi in ligand_map[c]])
                    print(f'  {c}: {tokens}')

            chain_ids = sorted(chain_ranges.keys())
            build_chain_buttons(chain_ids)
            if chain_ids:
                set_chain(chain_ids[0])
            else:
                selected_chain_text.value = ''
                chain_btns.children = tuple()
                with selector_out:
                    clear_output(wait=True)
                    print('No protein chains found in structure.')
                refresh_viewer()
        except Exception as e:
            print(f'Error: {e}')


for _button in (load_btn, dragmode_btn, clear_anchor_btn, apply_contigs_btn, clear_contigs_btn, clear_hotspots_btn, refresh_3d_btn):
    _reset_button_click_handlers(_button)


load_btn.on_click(on_load)
dragmode_btn.on_click(enable_box_select)
clear_anchor_btn.on_click(clear_anchor)
apply_contigs_btn.on_click(apply_contigs_from_text)
clear_contigs_btn.on_click(clear_contigs)
clear_hotspots_btn.on_click(clear_hotspots)
refresh_3d_btn.on_click(manual_refresh_viewer)

# Keep preview callbacks idempotent after function overrides.
try:
    hotspots_preview.unobserve(on_preview_change, names='value')
except Exception:
    pass
hotspots_preview.observe(on_preview_change, names='value')

try:
    show_ligands_chk.unobserve(on_preview_change, names='value')
except Exception:
    pass
show_ligands_chk.observe(on_preview_change, names='value')

In [11]:
# Display UI from Cell 5 so Cell 5 can stay collapsed
if 'viewer_ui' not in globals():
    raise RuntimeError('viewer_ui is not defined. Run Cell 5 first.')
display(viewer_ui)

## 2) Define design inputs and method selection

In [20]:
def validate_hotspots(h: str) -> str:
    h = h.strip()
    if not h:
        return ''
    toks = [x.strip() for x in h.split(',') if x.strip()]
    bad = [t for t in toks if not re.match(r'^[A-Za-z]\d+$', t)]
    if bad:
        raise ValueError(f'Invalid hotspots: {bad}. Expected e.g. A56,B112')
    return ','.join(toks)

def validate_contigs(contigs: str) -> str:
    c = contigs.strip()
    if not c:
        raise ValueError('Contigs cannot be empty.')
    if not (c.startswith('[') and c.endswith(']')):
        raise ValueError('Contigs should use RFdiffusion format, e.g. [A18-132/0 65-120].')
    return c

PRESETS = {
    'quick_test_base': {
        'rfd_n_designs': 500,
        'rfd_batch_size': 5,
        'rfd_filters': 'rg<22',
        'rfd_model_path': '${WF_PATH}/models/rfdiffusion/Complex_base_ckpt.pt',
        'pmpnn_weights': '${WF_PATH}/models/HyperMPNN/retrained_models/v48_020_epoch300_hyper.pt',
        'rfd_extra_args': 'potentials.guiding_potentials=[\\\"type:binder_ROG,weight:7,min_dist:10\\\"] potentials.guide_decay=\\\"quadratic\\\"',
        'pmpnn_seqs_per_struct': 1,
        'pmpnn_relax_cycles': 1,
        'af2ig_recycle': 3,
        'refold_af2ig_filters': 'pae_interaction<=10;plddt_binder>=80',
        'refold_max': 200,
        'bindcraft_n_traj': 500,
        'bindcraft_batch_size': 1,
        'bindcraft_advanced_settings_preset': 'default_4stage_multimer_hardtarget',
        'bindcraft_filters_preset': 'default_filters',
        'num_designs': 500,
        'batch_size': 5,
        'budget': 10
    },
    'quick_test_beta': {
        'rfd_n_designs': 500,
        'rfd_batch_size': 5,
        'rfd_filters': 'rg<22',
        'rfd_model_path': '${WF_PATH}/models/rfdiffusion/Complex_beta_ckpt.pt',
        'pmpnn_weights': '${WF_PATH}/models/HyperMPNN/retrained_models/v48_020_epoch300_hyper.pt',
        'rfd_extra_args': 'potentials.guiding_potentials=[\\\"type:binder_ROG,weight:7,min_dist:10\\\"] potentials.guide_decay=\\\"quadratic\\\"',
        'pmpnn_seqs_per_struct': 1,
        'pmpnn_relax_cycles': 1,
        'af2ig_recycle': 3,
        'refold_af2ig_filters': 'pae_interaction<=10;plddt_binder>=80',
        'refold_max': 200,
        'bindcraft_n_traj': 500,
        'bindcraft_batch_size': 1,
        'bindcraft_advanced_settings_preset': 'default_4stage_multimer_hardtarget',
        'bindcraft_filters_preset': 'default_filters',
        'num_designs': 500,
        'batch_size': 5,
        'budget': 10
    },
    'exploratory_large_base': {
        'rfd_n_designs': 5000,
        'rfd_batch_size': 5,
        'rfd_filters': 'rg<22',
        'rfd_model_path': '${WF_PATH}/models/rfdiffusion/Complex_base_ckpt.pt',
        'pmpnn_weights': '${WF_PATH}/models/HyperMPNN/retrained_models/v48_020_epoch300_hyper.pt',
        'rfd_extra_args': 'potentials.guiding_potentials=[\\\"type:binder_ROG,weight:7,min_dist:10\\\"] potentials.guide_decay=\\\"quadratic\\\"',
        'pmpnn_seqs_per_struct': 2,
        'pmpnn_relax_cycles': 5,
        'af2ig_recycle': 3,
        'refold_af2ig_filters': 'pae_interaction<=10;plddt_binder>=80',
        'refold_max': 200,
        'bindcraft_n_traj': 5000,
        'bindcraft_batch_size': 1,
        'bindcraft_advanced_settings_preset': 'default_4stage_multimer_hardtarget',
        'bindcraft_filters_preset': 'default_filters',
        'num_designs': 5000,
        'batch_size': 5,
        'budget': 100
    },
        'exploratory_large_beta': {
        'rfd_n_designs': 5000,
        'rfd_batch_size': 5,
        'rfd_filters': 'rg<22',
        'rfd_model_path': '${WF_PATH}/models/rfdiffusion/Complex_beta_ckpt.pt',
        'pmpnn_weights': '${WF_PATH}/models/HyperMPNN/retrained_models/v48_020_epoch300_hyper.pt',
        'rfd_extra_args': 'potentials.guiding_potentials=[\\\"type:binder_ROG,weight:7,min_dist:10\\\"] potentials.guide_decay=\\\"quadratic\\\"',
        'pmpnn_seqs_per_struct': 2,
        'pmpnn_relax_cycles': 5,
        'af2ig_recycle': 3,
        'refold_af2ig_filters': 'pae_interaction<=10;plddt_binder>=80',
        'refold_max': 200,
        'bindcraft_n_traj': 5000,
        'bindcraft_batch_size': 1,
        'bindcraft_advanced_settings_preset': 'default_4stage_multimer_hardtarget',
        'bindcraft_filters_preset': 'default_filters',
        'num_designs': 5000,
        'batch_size': 5,
        'budget': 100
    }    
}

print('Available sensible presets:')
for k in PRESETS:
    print(' -', k)

Available sensible presets:
 - quick_test_base
 - quick_test_beta
 - exploratory_large_base
 - exploratory_large_beta


In [21]:
# Edit these values before exporting
auto_chain = str(_state.get('selected_chain', '')).strip()
auto_hotspots = format_hotspots(_state.get('selected_hotspots', set())) if _state.get('selected_hotspots') else ''
auto_contigs = contigs_preview.value.strip() if 'contigs_preview' in globals() else ''
auto_ligands = ligands_preview.value.strip() if 'ligands_preview' in globals() else ''
auto_binder_len = binder_len_input.value.strip() if 'binder_len_input' in globals() else '65-120'

cfg = {
    'design_name': 'my_binder',
    'preset': 'quick_test_beta',  # choose from printed presets or customize values below
    'methods': ['rfd', 'bindcraft', 'boltzgen'],  # choose any subset
    'outdir': 'results',  # base output directory
    'profile': 'local',

    # Optional per-method outdir overrides. Leave empty to use:
    # results/rfd, results/bindcraft, results/boltzgen
    'rfd_outdir': '',
    'bindcraft_outdir': '',
    'boltzgen_outdir': '',

    # Shared biological inputs (pre-filled from viewer selections if present)
    'contigs': auto_contigs or '[A18-132/0 65-120]',
    'binder_len': auto_binder_len or '65-120',
    'hotspot_res': auto_hotspots or 'A56',
    'selected_ligands': auto_ligands,  # e.g. A177:HEM
    'target_chains': auto_chain or 'A',

    # Refold options (used by rfd / rfd_partial workflows)
    'enable_boltz2_refold': True,
    'refold_use_msa_server': True,
    'refold_target_fasta': '',  # full-length target FASTA path if available
    'refold_target_templates': '',  # defaulted to inputs/reference.pdb if left empty

    # Optional FoldSeek
    'do_foldseek': False,
    'foldseek_database': 'CATH50',
    'foldseek_databases_path': ''
}

cfg

{'design_name': 'my_binder',
 'preset': 'quick_test_beta',
 'methods': ['rfd', 'bindcraft', 'boltzgen'],
 'outdir': 'results',
 'profile': 'local',
 'rfd_outdir': '',
 'bindcraft_outdir': '',
 'boltzgen_outdir': '',
 'contigs': '[R26-42/R81-113/R157-204/R454-486/0 55-110]',
 'binder_len': '55-110',
 'hotspot_res': 'R188,R465,R477',
 'selected_ligands': 'R602:ACH',
 'target_chains': 'A',
 'enable_boltz2_refold': True,
 'refold_use_msa_server': True,
 'refold_target_fasta': '',
 'refold_target_templates': '',
 'do_foldseek': False,
 'foldseek_database': 'CATH50',
 'foldseek_databases_path': ''}

## 3) Export nf-binder-design config files

In [22]:
missing_symbols = [
    name for name in ('validate_hotspots', 'validate_contigs', 'PRESETS', 'cfg')
    if name not in globals()
]
if missing_symbols:
    raise RuntimeError(
        'Missing notebook state: ' + ', '.join(missing_symbols) + '. '
        'Run Cell 8 (presets/validators) and Cell 9 (cfg) before this export cell.'
    )

ORIGINAL_PDB_REL = 'inputs/target_original.pdb'
TRUNCATED_PDB_REL = 'inputs/target_truncated_selected.pdb'
REFERENCE_PDB_REL = 'inputs/reference.pdb'

METHOD_DIRS = {
    'rfd': OUTDIR / 'rfd',
    'bindcraft': OUTDIR / 'bindcraft',
    'boltzgen': OUTDIR / 'boltzgen',
}

base_outdir = str(cfg.get('outdir', 'results')).rstrip('/')
rfd_outdir = str(cfg.get('rfd_outdir') or f'{base_outdir}/rfd').rstrip('/')
bindcraft_outdir = str(cfg.get('bindcraft_outdir') or f'{base_outdir}/bindcraft').rstrip('/')
boltzgen_outdir = str(cfg.get('boltzgen_outdir') or f'{base_outdir}/boltzgen').rstrip('/')


def parse_rf_contigs(contigs: str) -> Tuple[List[Tuple[str, int, int]], str]:
    c = validate_contigs(contigs)
    body = c.strip()[1:-1].strip()
    if not body:
        return [], ''
    parts = body.split()
    binder_len = ''
    receptor_tokens = parts
    if parts and re.match(r'^\d+-\d+$', parts[-1].strip()):
        binder_len = parts[-1].strip()
        receptor_tokens = parts[:-1]
    receptor_part = '/'.join(t.strip() for t in receptor_tokens)
    segments: List[Tuple[str, int, int]] = []
    for frag in receptor_part.split('/'):
        f = frag.strip()
        if not f or f == '0':
            continue
        m = re.match(r'^([A-Za-z])(\d+)(?:-(\d+))?$', f)
        if not m:
            continue
        chain = m.group(1)
        a = int(m.group(2))
        b = int(m.group(3)) if m.group(3) else a
        if a > b:
            a, b = b, a
        segments.append((chain, a, b))
    return segments, binder_len


def parse_ligands_text(text: str, allowed_chains: Set[str]) -> Set[Tuple[str, int]]:
    out: Set[Tuple[str, int]] = set()
    for token in [x.strip() for x in text.split(',') if x.strip()]:
        m = re.match(r'^([A-Za-z])(\d+):([A-Za-z0-9_\-]+)$', token)
        if not m:
            continue
        chain = m.group(1)
        resi = int(m.group(2))
        resn = m.group(3).upper()
        if chain in allowed_chains and resn not in WATER_NAMES:
            out.add((resn, resi))
    return out


def collect_selected_ligands(allowed_chains: Set[str]) -> Set[Tuple[str, int]]:
    selected: Set[Tuple[str, int]] = set()
    for token in _state.get('selected_contig_ligands', set()):
        parsed = parse_ligand_token(token)
        if parsed is None:
            continue
        chain, resn, resi = parsed
        if chain in allowed_chains and resn not in WATER_NAMES:
            selected.add((resn, int(resi)))
    if cfg.get('selected_ligands'):
        selected.update(parse_ligands_text(str(cfg['selected_ligands']), allowed_chains))
    return selected


def collect_selected_ligands(allowed_chains: Set[str]) -> Set[Tuple[str, int]]:
    selected: Set[Tuple[str, int]] = set()
    for token in _state.get('selected_contig_ligands', set()):
        parsed = parse_ligand_token(token)
        if parsed is None:
            continue
        chain, resn, resi = parsed
        if chain in allowed_chains and resn not in WATER_NAMES:
            selected.add((resn, int(resi)))
    if cfg.get('selected_ligands'):
        selected.update(parse_ligands_text(str(cfg['selected_ligands']), allowed_chains))
    return selected


def build_truncated_selected_pdb(
    source_pdb: Path,
    out_pdb: Path,
    keep_residues_by_chain: Dict[str, Set[int]],
    keep_ligands_by_chain: Dict[str, Set[Tuple[str, int]]],
    chain_priority: Optional[List[str]] = None,
) -> Dict[str, str]:
    if not source_pdb.exists():
        raise FileNotFoundError(f'Input PDB not found: {source_pdb}')

    kept_chain_ids = sorted(set(keep_residues_by_chain.keys()) | set(keep_ligands_by_chain.keys()))
    if not kept_chain_ids:
        raise ValueError('No chains selected for truncation.')

    ordered_chains: List[str] = []
    if chain_priority:
        for chain in chain_priority:
            if chain in kept_chain_ids and chain not in ordered_chains:
                ordered_chains.append(chain)
    for chain in kept_chain_ids:
        if chain not in ordered_chains:
            ordered_chains.append(chain)

    chain_ids = list('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789')
    if len(ordered_chains) > len(chain_ids):
        raise ValueError('Too many chains selected to remap uniquely.')
    chain_map = {old: chain_ids[idx] for idx, old in enumerate(ordered_chains)}

    kept_lines: List[str] = []
    for raw in source_pdb.read_text().splitlines():
        if len(raw) < 26:
            continue
        rec = raw[:6].strip()
        if rec not in {'ATOM', 'HETATM', 'ANISOU'}:
            continue

        chain = raw[21].strip()
        if chain not in chain_map:
            continue

        resn = raw[17:20].strip().upper()
        try:
            resi = int(raw[22:26].strip())
        except Exception:
            continue

        keep = False
        if rec in {'ATOM', 'ANISOU'}:
            keep = resi in keep_residues_by_chain.get(chain, set())
        else:
            if resn in WATER_NAMES:
                keep = False
            else:
                keep = (resn, resi) in keep_ligands_by_chain.get(chain, set())

        if not keep:
            continue

        kept_lines.append(raw[:21] + chain_map[chain] + raw[22:])

    if not kept_lines:
        raise ValueError('Truncated structure is empty. Check contigs and selected chain/ligands.')

    out_pdb.write_text('\n'.join(kept_lines) + '\nEND\n')
    return chain_map


def build_reference_selected_chains_pdb(
    source_pdb: Path,
    out_pdb: Path,
    selected_chains: Set[str],
    chain_map: Dict[str, str],
):
    if not source_pdb.exists():
        raise FileNotFoundError(f'Input PDB not found: {source_pdb}')
    if not selected_chains:
        raise ValueError('No chains selected to build reference template.')

    kept_lines: List[str] = []
    for raw in source_pdb.read_text().splitlines():
        if len(raw) < 26:
            continue
        rec = raw[:6].strip()
        if rec not in {'ATOM', 'HETATM', 'ANISOU'}:
            continue

        chain = raw[21].strip()
        if chain not in selected_chains:
            continue
        if chain not in chain_map:
            continue

        kept_lines.append(raw[:21] + chain_map[chain] + raw[22:])

    if not kept_lines:
        raise ValueError('Reference structure is empty. Check selected contig chains.')

    out_pdb.write_text('\n'.join(kept_lines) + '\nEND\n')


def build_truncated_chain_a_pdb(
    source_pdb: Path,
    out_pdb: Path,
    source_chain: str,
    keep_residues: Set[int],
    keep_ligands: Set[Tuple[str, int]],
):
    if not source_pdb.exists():
        raise FileNotFoundError(f'Input PDB not found: {source_pdb}')

    kept_lines: List[str] = []
    for raw in source_pdb.read_text().splitlines():
        if len(raw) < 26:
            continue
        rec = raw[:6].strip()
        if rec not in {'ATOM', 'HETATM', 'ANISOU'}:
            continue

        chain = raw[21].strip()
        if chain not in keep_residues_by_chain and chain not in keep_ligands_by_chain:
            continue

        resn = raw[17:20].strip().upper()
        try:
            resi = int(raw[22:26].strip())
        except Exception:
            continue

        keep = False
        if rec in {'ATOM', 'ANISOU'}:
            keep = resi in keep_residues_by_chain.get(chain, set())
        else:
            if resn in WATER_NAMES:
                keep = False
            else:
                keep = (resn, resi) in keep_ligands_by_chain.get(chain, set())

        if not keep:
            continue

        kept_lines.append(raw)

    if not kept_lines:
        raise ValueError('Truncated structure is empty. Check contigs and selected chain/ligands.')

    out_pdb.write_text('\n'.join(kept_lines) + '\nEND\n')


def format_selected_contigs(binder_len: str) -> str:
    contigs = format_contigs_from_selection(binder_len)
    if not contigs:
        raise ValueError('No receptor residues available to build contigs.')
    return contigs


def write_json(path: Path, obj: dict):
    path.write_text(json.dumps(obj, indent=2))


def make_cmd(params_file: Path, profile: str) -> str:
    return f"nextflow run Australian-Protein-Design-Initiative/nf-binder-design -params-file {params_file} -profile {profile} -resume"


def write_local_script(method: str, method_dir: Path, params_name: str, profile: str) -> Path:
    method_dir.mkdir(parents=True, exist_ok=True)
    script_path = method_dir / f'run_{method}_local.sh'
    body = '\n'.join([
        '#!/usr/bin/env bash',
        'set -euo pipefail',
        '',
        'SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"',
        'METHOD_DIR="$(cd "${SCRIPT_DIR}" && pwd)"',
        'cd "${METHOD_DIR}"',
        '',
        f'nextflow run Australian-Protein-Design-Initiative/nf-binder-design -params-file {params_name} -profile {profile} -resume',
        ''
    ])
    script_path.write_text(body)
    script_path.chmod(0o755)
    return script_path


common_inputs_dir = OUTDIR / 'inputs'
common_inputs_dir.mkdir(parents=True, exist_ok=True)

original_pdb_path = common_inputs_dir / 'target_original.pdb'
raw_pdb_path = str(_state.get('pdb_path', '')).strip()
source_pdb = Path(raw_pdb_path) if raw_pdb_path else None

if source_pdb is not None and source_pdb.exists() and source_pdb.is_file():
    original_pdb_path.write_text(source_pdb.read_text())
elif str(_state.get('pdb_text', '')).strip():
    original_pdb_path.write_text(str(_state['pdb_text']))
elif original_pdb_path.exists() and original_pdb_path.is_file():
    pass
else:
    raise RuntimeError('Target PDB is unavailable. Run Cell 5 (Load + View) before exporting.')

cfg_contigs = str(cfg.get('contigs', '')).strip() or '[A18-132/0 65-120]'
export_contigs_by_chain = selected_contigs_by_chain_as_ranges() if 'selected_contigs_by_chain_as_ranges' in globals() else {}
if not export_contigs_by_chain:
    export_contigs_by_chain = parse_contigs_for_all_chains(cfg_contigs) if 'parse_contigs_for_all_chains' in globals() else {}
if not export_contigs_by_chain:
    raise ValueError('No selected residues found for truncation. Define contigs first.')

_, binder_len_from_contigs = parse_rf_contigs(cfg_contigs)
cfg_binder_len = str(cfg.get('binder_len', '')).strip()
if cfg_binder_len and not re.match(r'^\d+(?:-\d+)?$', cfg_binder_len):
    raise ValueError('cfg[binder_len] must be an integer or range like 65-120.')

export_residues_by_chain: Dict[str, Set[int]] = {}
for chain_id, chain_values in export_contigs_by_chain.items():
    residues: Set[int] = set()
    if chain_values and isinstance(next(iter(chain_values)), tuple):
        for a, b in chain_values:
            residues.update(range(int(a), int(b) + 1))
    else:
        residues.update(int(r) for r in chain_values)
    if residues:
        export_residues_by_chain[chain_id] = residues

selected_chain_ids = sorted(export_residues_by_chain.keys())
binder_len = cfg_binder_len or binder_len_input.value.strip() or binder_len_from_contigs or '65-120'
export_contigs = format_selected_contigs(binder_len)
selected_hotspots = format_hotspots(_state.get('selected_hotspots', set())) or validate_hotspots(str(cfg.get('hotspot_res', '')).strip())

hotspot_counts_by_chain: Dict[str, int] = {}
for token in [x.strip() for x in selected_hotspots.split(',') if x.strip()]:
    m = re.match(r'^([A-Za-z0-9])(-?\d+)$', token)
    if not m:
        continue
    h_chain = m.group(1)
    if h_chain in selected_chain_ids:
        hotspot_counts_by_chain[h_chain] = hotspot_counts_by_chain.get(h_chain, 0) + 1

chain_priority = sorted(
    selected_chain_ids,
    key=lambda c: (
        -(len(export_residues_by_chain.get(c, set())) + hotspot_counts_by_chain.get(c, 0)),
        -len(export_residues_by_chain.get(c, set())),
        -hotspot_counts_by_chain.get(c, 0),
        c,
    ),
)

selected_ligands_by_chain: Dict[str, Set[Tuple[str, int]]] = {}
for token in _state.get('selected_contig_ligands', set()):
    parsed = parse_ligand_token(token)
    if parsed is None:
        continue
    chain, resn, resi = parsed
    if chain in selected_chain_ids and resn not in WATER_NAMES:
        selected_ligands_by_chain.setdefault(chain, set()).add((resn, int(resi)))

if cfg.get('selected_ligands'):
    for token in [x.strip() for x in str(cfg['selected_ligands']).split(',') if x.strip()]:
        parsed = parse_ligand_token(token)
        if parsed is None:
            continue
        chain, resn, resi = parsed
        if chain in selected_chain_ids and resn not in WATER_NAMES:
            selected_ligands_by_chain.setdefault(chain, set()).add((resn, int(resi)))

truncated_pdb_path = common_inputs_dir / 'target_truncated_selected.pdb'
chain_map = build_truncated_selected_pdb(
    source_pdb=original_pdb_path,
    out_pdb=truncated_pdb_path,
    keep_residues_by_chain=export_residues_by_chain,
    keep_ligands_by_chain=selected_ligands_by_chain,
    chain_priority=chain_priority,
)

reference_pdb_path = common_inputs_dir / 'reference.pdb'
build_reference_selected_chains_pdb(
    source_pdb=original_pdb_path,
    out_pdb=reference_pdb_path,
    selected_chains=set(export_residues_by_chain.keys()),
    chain_map=chain_map,
)


def remap_chain_prefixed_tokens(text: str, chain_map: Dict[str, str]) -> str:
    if not text:
        return text
    return re.sub(
        r'(?<![A-Za-z0-9])([A-Za-z0-9])(?=\d)',
        lambda m: chain_map.get(m.group(1), m.group(1)),
        text,
    )


selected_chain_ids = [new_chain for _, new_chain in sorted(chain_map.items(), key=lambda kv: kv[1])]
remapped_residues_by_chain = {chain_map[old_chain]: set(residues) for old_chain, residues in export_residues_by_chain.items() if old_chain in chain_map}
export_contigs = remap_chain_prefixed_tokens(export_contigs, chain_map)
selected_hotspots = ','.join(
    [
        (chain_map.get(tok[0], tok[0]) + tok[1:]) if re.match(r'^[A-Za-z0-9]-?\d+$', tok) else tok
        for tok in [x.strip() for x in selected_hotspots.split(',') if x.strip()]
    ]
)

print(f'Saved shared original template PDB: {common_inputs_dir / "target_original.pdb"}')
print(f'Saved shared truncated input PDB:  {common_inputs_dir / "target_truncated_selected.pdb"}')
print(f'Saved shared full-chain reference PDB: {reference_pdb_path}')
print(f'Export contigs: {export_contigs}')
print(f'Default outdirs -> rfd: {rfd_outdir}, bindcraft: {bindcraft_outdir}, boltzgen: {boltzgen_outdir}')
if selected_ligands_by_chain:
    lig_txt = ', '.join([f'{chain}{resi}:{resn}' for chain in sorted(selected_ligands_by_chain.keys()) for resn, resi in sorted(selected_ligands_by_chain[chain], key=lambda x: (x[1], x[0]))])
    print(f'Selected ligands in truncated input: {lig_txt}')

p = PRESETS[cfg['preset']]
print(f"Using preset: {cfg['preset']}")


def build_rfd_params(cfg: dict, p: dict) -> dict:
    out = {
        'method': 'rfd',
        'input_pdb': TRUNCATED_PDB_REL,
        'outdir': rfd_outdir,
        'design_name': cfg['design_name'],
        'contigs': validate_contigs(export_contigs),
        'hotspot_res': selected_hotspots,
        'rfd_n_designs': p['rfd_n_designs'],
        'rfd_batch_size': p['rfd_batch_size'],
        'rfd_filters': p['rfd_filters'],
        'pmpnn_seqs_per_struct': p['pmpnn_seqs_per_struct'],
        'pmpnn_relax_cycles': p['pmpnn_relax_cycles'],
        'af2ig_recycle': p['af2ig_recycle'],
    }
    if p.get('rfd_model_path'):
        out['rfd_model_path'] = p['rfd_model_path']
    if p.get('pmpnn_weights'):
        out['pmpnn_weights'] = p['pmpnn_weights']
    if p.get('rfd_extra_args'):
        out['rfd_extra_args'] = p['rfd_extra_args']
    if cfg['enable_boltz2_refold']:
        out['refold_af2ig_filters'] = p['refold_af2ig_filters']
        out['refold_max'] = p['refold_max']
        out['refold_use_msa_server'] = cfg['refold_use_msa_server']
        if cfg['refold_target_fasta']:
            out['refold_target_fasta'] = cfg['refold_target_fasta']
        out['refold_target_templates'] = cfg.get('refold_target_templates') or REFERENCE_PDB_REL
    if cfg['do_foldseek']:
        out['do_foldseek'] = True
        out['foldseek_database'] = cfg['foldseek_database']
        if cfg['foldseek_databases_path']:
            out['foldseek_databases_path'] = cfg['foldseek_databases_path']
    return out


def build_bindcraft_params(cfg: dict, p: dict) -> dict:
    out = {
        'method': 'bindcraft',
        'input_pdb': TRUNCATED_PDB_REL,
        'outdir': bindcraft_outdir,
        'design_name': cfg['design_name'],
        'hotspot_res': validate_hotspots(selected_hotspots) if selected_hotspots else '',
        'binder_length_range': binder_len if binder_len else '55-120',
        'bindcraft_n_traj': p['bindcraft_n_traj'],
        'bindcraft_batch_size': p['bindcraft_batch_size'],
        'bindcraft_advanced_settings_preset': p['bindcraft_advanced_settings_preset'],
        'bindcraft_filters_preset': p['bindcraft_filters_preset']
    }
    out['contigs'] = validate_contigs(export_contigs)
    if cfg['do_foldseek']:
        out['do_foldseek'] = True
        out['foldseek_database'] = cfg['foldseek_database']
        if cfg['foldseek_databases_path']:
            out['foldseek_databases_path'] = cfg['foldseek_databases_path']
    return out


def format_boltzgen_sequence_range(binder_len_value: str) -> str:
    text = str(binder_len_value).strip()
    if re.match(r'^\d+-\d+$', text):
        lo, hi = text.split('-')
        return f'{int(lo)}..{int(hi)}'
    if re.match(r'^\d+$', text):
        num = int(text)
        return f'{num}..{num}'
    return '55..120'


def format_boltzgen_res_index(residues: Set[int]) -> str:
    if not residues:
        return ''
    vals = sorted(int(r) for r in residues)
    ranges: List[str] = []
    start = vals[0]
    prev = vals[0]
    for resi in vals[1:]:
        if resi == prev + 1:
            prev = resi
            continue
        ranges.append(f'{start}..{prev}' if start != prev else str(start))
        start = resi
        prev = resi
    ranges.append(f'{start}..{prev}' if start != prev else str(start))
    return ','.join(ranges)


def build_boltzgen_index_map(residues_by_chain: Dict[str, Set[int]]) -> Dict[str, Dict[int, int]]:
    out: Dict[str, Dict[int, int]] = {}
    for chain_id, residues in residues_by_chain.items():
        ordered = sorted(int(r) for r in residues)
        out[chain_id] = {orig_resi: idx + 1 for idx, orig_resi in enumerate(ordered)}
    return out


def next_boltzgen_binder_chain(chain_ids: List[str]) -> str:
    used = set(chain_ids)
    for chain_id in list('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789'):
        if chain_id not in used:
            return chain_id
    raise ValueError('Unable to assign a unique binder chain ID for BoltzGen YAML.')


def build_boltzgen_yaml(cfg: dict, p: dict) -> dict:
    binder_chain_id = next_boltzgen_binder_chain(selected_chain_ids)
    boltzgen_index_map = build_boltzgen_index_map(remapped_residues_by_chain)
    include_entries = []
    for chain_id in selected_chain_ids:
        chain_entry = {'id': chain_id}
        local_positions = set(boltzgen_index_map.get(chain_id, {}).values())
        res_index = format_boltzgen_res_index(local_positions)
        if res_index:
            chain_entry['res_index'] = res_index
        include_entries.append({'chain': chain_entry})

    binding_entries = []
    hotspot_map: Dict[str, List[int]] = {}
    for token in [x.strip() for x in selected_hotspots.split(',') if x.strip()]:
        m = re.match(r'^([A-Za-z0-9])(-?\d+)$', token)
        if not m:
            continue
        hotspot_map.setdefault(m.group(1), []).append(int(m.group(2)))
    for chain_id in selected_chain_ids:
        residues = hotspot_map.get(chain_id, [])
        if residues:
            mapped_residues = [boltzgen_index_map[chain_id][resi] for resi in residues if resi in boltzgen_index_map.get(chain_id, {})]
            if mapped_residues:
                binding_entries.append({'chain': {'id': chain_id, 'binding': ','.join(str(resi) for resi in mapped_residues)}})

    file_entity = {
        'path': TRUNCATED_PDB_REL,
        'include': include_entries,
        'structure_groups': 'all',
    }
    if binding_entries:
        file_entity['binding_types'] = binding_entries

    return {
        'entities': [
            {'protein': {'id': binder_chain_id, 'sequence': format_boltzgen_sequence_range(binder_len)}},
            {'file': file_entity},
        ]
    }


def build_boltzgen_params(cfg: dict, p: dict, yaml_path: str) -> dict:
    out = {
        'method': 'boltzgen',
        'config_yaml': yaml_path,
        'outdir': boltzgen_outdir,
        'design_name': cfg['design_name'],
        'protocol': 'protein-anything',
        'num_designs': p['num_designs'],
        'batch_size': p['batch_size'],
        'budget': p['budget']
    }
    if cfg['do_foldseek']:
        out['do_foldseek'] = True
        out['foldseek_database'] = cfg['foldseek_database']
        if cfg['foldseek_databases_path']:
            out['foldseek_databases_path'] = cfg['foldseek_databases_path']
    return out


generated = []
local_scripts = []

if 'rfd' in cfg['methods']:
    method_dir = METHOD_DIRS['rfd']
    method_inputs = method_dir / 'inputs'
    method_inputs.mkdir(parents=True, exist_ok=True)
    method_inputs.joinpath('target_original.pdb').write_text(original_pdb_path.read_text())
    method_inputs.joinpath('target_truncated_selected.pdb').write_text(truncated_pdb_path.read_text())
    method_inputs.joinpath('reference.pdb').write_text(reference_pdb_path.read_text())
    legacy_path = method_inputs / 'target_truncated_chainA.pdb'
    if legacy_path.exists():
        legacy_path.unlink()

    rfd = build_rfd_params(cfg, p)
    rfd_path = method_dir / 'params.rfd.json'
    write_json(rfd_path, rfd)
    generated.append(rfd_path)
    local_scripts.append(write_local_script('rfd', method_dir, 'params.rfd.json', cfg['profile']))

if 'bindcraft' in cfg['methods']:
    method_dir = METHOD_DIRS['bindcraft']
    method_inputs = method_dir / 'inputs'
    method_inputs.mkdir(parents=True, exist_ok=True)
    method_inputs.joinpath('target_original.pdb').write_text(original_pdb_path.read_text())
    method_inputs.joinpath('target_truncated_selected.pdb').write_text(truncated_pdb_path.read_text())
    method_inputs.joinpath('reference.pdb').write_text(reference_pdb_path.read_text())
    legacy_path = method_inputs / 'target_truncated_chainA.pdb'
    if legacy_path.exists():
        legacy_path.unlink()

    bc = build_bindcraft_params(cfg, p)
    bc_path = method_dir / 'params.bindcraft.json'
    write_json(bc_path, bc)
    generated.append(bc_path)
    local_scripts.append(write_local_script('bindcraft', method_dir, 'params.bindcraft.json', cfg['profile']))

if 'boltzgen' in cfg['methods']:
    method_dir = METHOD_DIRS['boltzgen']
    method_inputs = method_dir / 'inputs'
    method_inputs.mkdir(parents=True, exist_ok=True)
    method_inputs.joinpath('target_original.pdb').write_text(original_pdb_path.read_text())
    method_inputs.joinpath('target_truncated_selected.pdb').write_text(truncated_pdb_path.read_text())
    method_inputs.joinpath('reference.pdb').write_text(reference_pdb_path.read_text())
    legacy_path = method_inputs / 'target_truncated_chainA.pdb'
    if legacy_path.exists():
        legacy_path.unlink()

    bg_yaml = build_boltzgen_yaml(cfg, p)
    bg_yaml_path = method_dir / f"{cfg['design_name']}.boltzgen.yaml"
    bg_yaml_path.write_text(yaml.safe_dump(bg_yaml, sort_keys=False))

    bg = build_boltzgen_params(cfg, p, bg_yaml_path.name)
    bg_path = method_dir / 'params.boltzgen.json'
    write_json(bg_path, bg)
    generated.extend([bg_yaml_path, bg_path])
    local_scripts.append(write_local_script('boltzgen', method_dir, 'params.boltzgen.json', cfg['profile']))

for f in generated:
    print(f'Wrote: {f}')
for s in local_scripts:
    print(f'Wrote script: {s}')

cmds_path = OUTDIR / 'run_commands.sh'
cmd_lines = ['#!/usr/bin/env bash', 'set -euo pipefail', '']
for method in cfg.get('methods', []):
    if method in METHOD_DIRS:
        cmd_lines.append(f'bash "{METHOD_DIRS[method] / f"run_{method}_local.sh"}"')
cmds_path.write_text('\n'.join(cmd_lines) + '\n')
print(f'Wrote: {cmds_path}')

Saved shared original template PDB: /home/jmobbs/code/nf_binder_project/nfb_setup_output/inputs/target_original.pdb
Saved shared truncated input PDB:  /home/jmobbs/code/nf_binder_project/nfb_setup_output/inputs/target_truncated_selected.pdb
Saved shared full-chain reference PDB: /home/jmobbs/code/nf_binder_project/nfb_setup_output/inputs/reference.pdb
Export contigs: [A26-42/A81-113/A157-204/A454-486/0 55-110]
Default outdirs -> rfd: results/rfd, bindcraft: results/bindcraft, boltzgen: results/boltzgen
Selected ligands in truncated input: R602:ACH
Using preset: quick_test_beta
Wrote: /home/jmobbs/code/nf_binder_project/nfb_setup_output/rfd/params.rfd.json
Wrote: /home/jmobbs/code/nf_binder_project/nfb_setup_output/bindcraft/params.bindcraft.json
Wrote: /home/jmobbs/code/nf_binder_project/nfb_setup_output/boltzgen/my_binder.boltzgen.yaml
Wrote: /home/jmobbs/code/nf_binder_project/nfb_setup_output/boltzgen/params.boltzgen.json
Wrote script: /home/jmobbs/code/nf_binder_project/nfb_setup_o

## 4) Optional HPC launch script generation (custom profiles)

Use this section if you want cluster-ready launch scripts (for example SLURM/M3) using your own Nextflow profile/config.

It writes scripts into each method folder: nfb_setup_output/rfd/, nfb_setup_output/bindcraft/, nfb_setup_output/boltzgen/.
Set `hpc_cfg['enabled'] = True` before running.

In [23]:
hpc_cfg = {
    'enabled': True,
    'cluster_name': 'm3',
    'wf_path': '/fs04/scratch2/nx54/jmobbs/software/nf_binder/nf-binder-design/',
    'profile_config': '${WF_PATH}/conf/platforms/m3.m3t007.config',
    'module_load': 'module load nextflow/24.04.3 || true',
    'tmpdir': '/fs04/scratch2/nx54/jmobbs/tmp',
    'apptainer_cache': '/fs04/scratch2/nx54/jmobbs/software/tmp/apptainer/nxf_cache/',
    'logs_dir': 'results/logs',
    'extra_nextflow_args': '',
}

method_param_map = {
    'rfd': 'params.rfd.json',
    'bindcraft': 'params.bindcraft.json',
    'boltzgen': 'params.boltzgen.json',
}

method_dirs = {
    'rfd': OUTDIR / 'rfd',
    'bindcraft': OUTDIR / 'bindcraft',
    'boltzgen': OUTDIR / 'boltzgen',
}

selected_methods = [m for m in cfg.get('methods', []) if m in method_param_map]

if not hpc_cfg['enabled']:
    print('HPC script generation is disabled.')
    print('Set hpc_cfg[\'enabled\'] = True and re-run this cell to write scripts.')
else:
    wf_path = hpc_cfg['wf_path']
    profile_config = hpc_cfg['profile_config']
    logs_dir = hpc_cfg['logs_dir']

    def render_hpc_script(method: str, params_file: str) -> str:
        lines = [
            '#!/bin/bash',
            f'# adapted for {hpc_cfg["cluster_name"]} HPC',
            '# configured for scheduler execution',
            'set -euo pipefail',
            '',
            '# CHANGE THIS: path to your nf-binder-design clone',
            f'export WF_PATH="{wf_path}"',
            '',
            'SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"',
            'METHOD_DIR="$(cd "${SCRIPT_DIR}" && pwd)"',
            'cd "${METHOD_DIR}"',
            '',
            f'mkdir -p {logs_dir}',
            'DATESTAMP=$(date +%Y%m%d_%H%M%S)',
            '',
            '# temp directories',
            f'export TMPDIR="{hpc_cfg["tmpdir"]}"',
            'export NXF_TEMP="$TMPDIR"',
            'export NXF_APPTAINER_TMPDIR="$TMPDIR"',
            'mkdir -p "$TMPDIR"',
            '',
            '# Apptainer container cache',
            f'export NXF_APPTAINER_CACHEDIR="{hpc_cfg["apptainer_cache"]}"',
            '',
            '# Optional module load for cluster environments',
            hpc_cfg['module_load'],
            '',
            'PARAMS_FILE="' + params_file + '"',
            'if [[ "' + method + '" == "rfd" ]]; then',
            '  if command -v envsubst >/dev/null 2>&1; then',
            '    envsubst < "${PARAMS_FILE}" > "params_resolved.json"',
            '    PARAMS_FILE="params_resolved.json"',
            '  else',
            '    echo "Warning: envsubst not found; using unresolved params file."',
            '  fi',
            'fi',
            '',
            'nextflow \\',
            f'  -c "{profile_config}" run \\',
            '  "${WF_PATH}/main.nf" \\',
            '  -params-file "${PARAMS_FILE}" \\',
            '  -resume \\',
            f'  -with-report "{logs_dir}/report_{method}_${{DATESTAMP}}.html" \\',
            f'  -with-trace "{logs_dir}/trace_{method}_${{DATESTAMP}}.txt"',
        ]

        extra = str(hpc_cfg.get('extra_nextflow_args', '')).strip()
        if extra:
            lines[-1] = lines[-1] + ' \\'
            lines.append(f'  {extra}')

        return '\n'.join(lines) + '\n'

    written = []
    for method in selected_methods:
        method_dir = method_dirs[method]
        method_dir.mkdir(parents=True, exist_ok=True)
        params_file = method_param_map[method]
        script_path = method_dir / f'run_{method}_hpc.sh'
        script_path.write_text(render_hpc_script(method, params_file))
        script_path.chmod(0o755)
        written.append(script_path)

    if not written:
        print('No method scripts written (check cfg[\'methods\']).')
    else:
        print('Wrote HPC launch scripts:')
        for p in written:
            print(f' - {p}')

Wrote HPC launch scripts:
 - /home/jmobbs/code/nf_binder_project/nfb_setup_output/rfd/run_rfd_hpc.sh
 - /home/jmobbs/code/nf_binder_project/nfb_setup_output/bindcraft/run_bindcraft_hpc.sh
 - /home/jmobbs/code/nf_binder_project/nfb_setup_output/boltzgen/run_boltzgen_hpc.sh


In [24]:
print('Recommended next steps:')
print('1) Run the load/view cell and verify chain IDs + residue numbering.')
print('2) Set cfg values (contigs/hotspots/methods/preset).')
print('3) Run export cell to generate params and commands.')
print('4) Inspect nfb_setup_output/run_commands.sh and execute the method(s) you want.')

Recommended next steps:
1) Run the load/view cell and verify chain IDs + residue numbering.
2) Set cfg values (contigs/hotspots/methods/preset).
3) Run export cell to generate params and commands.
4) Inspect nfb_setup_output/run_commands.sh and execute the method(s) you want.
